# Fragrantica Dominant Accord Analysis

## Objective

This notebook analyzes the ordering of fragrance accords in the Fragrantica dataset to identify dominant accords and the most common combinations of the first two and first three accords. It also examines relationships between community engagement and fragrance ratings, validates statistical findings, and segments dominant accord pairs by popularity and rating.

Unlike an accord prevalence analysis, which examines all accords associated with each fragrance without considering their position, this analysis preserves the original ordering of accords. The first accord is treated as the **dominant accord**, followed by the secondary and tertiary accords, subject to validation of the source ordering.

## Analytical Scope

- **Dominant accord distribution:** Identify the most frequently occurring first accords across fragrances.
- **Top-two accord combinations:** Identify the most common ordered pairs of dominant and secondary accords.
- **Top-three accord combinations (triads):** Identify the most common ordered combinations of dominant, secondary, and tertiary accords.
- **Statistical validation:** Evaluate relationships between community vote counts and fragrance rankings or ratings, including the statistical significance of observed correlations.
- **Popularity and rating segmentation:** Classify dominant accord pairs into four segments based on their relative popularity and average rating, distinguishing pairs that perform relatively well on both dimensions from those with contrasting results.
- **Data preparation for recommendation systems:** Establish structured fragrance profiles that can support future fragrance wardrobe analysis, preference profiling, and personalized recommendations.

## Data Source

The analysis uses the cleaned Fragrantica reference dataset stored in Databricks:

`main.fragrance_analytics.fragrantica_perfumes`

The original source data remains unchanged. Extracted accord positions, normalized values, and analytical results are created as derived data.

## Methodology and Validation

The notebook validates the source representation and ordering of accords before interpreting the first accord as dominant. It then extracts and analyzes the first three accord positions while checking for missing values, duplicate records, and inconsistent accord representations.

The analytical workflow includes:

1. Calculating dominant accord frequencies and the distributions of ordered pairs and triads.
2. Evaluating the relationship between community vote counts and fragrance ratings or rankings using statistical correlation analysis.
3. Assessing statistical significance alongside correlation direction and strength, avoiding the interpretation of statistical significance as evidence of a strong relationship or causation.
4. Aggregating ordered dominant accord pairs to calculate fragrance counts, vote-based popularity metrics, and rating statistics.
5. Segmenting eligible accord pairs into four groups using the median popularity and median rating across the analyzed pairs as reference thresholds.
6. Interpreting results in the context of sample sizes and the limitations of community-generated ratings and vote counts.

The pair segmentation is relative to the eligible pairs included in the analysis. Popularity is represented by community engagement metrics rather than direct sales data.

## Key Findings and Expected Outcomes

The analysis identified a **statistically significant, weak negative correlation** between the number of community votes and fragrance ranking, indicating a small inverse relationship in the analyzed data. This finding is associational and does not establish causation.

The notebook also provides a structured view of dominant accord frequencies, common ordered pairs and triads, and a four-quadrant segmentation of dominant accord pairs by relative popularity and rating.

Together, these outputs help identify common fragrance accord structures, explore differences in community engagement and ratings, and establish a foundation for subsequent dominant-accord rating analysis, gender comparisons in a separate notebook, and future fragrance recommendation functionality.

In [0]:
%pip install --upgrade "plotly>=6.1.1" "kaleido>=1.0.0"

from pyspark.sql import functions as F

# Load the cleaned Fragrantica reference dataset
source_table = "main.fragrance_analytics.fragrantica_perfumes"
df = spark.table(source_table)

# Exclude fragrances without accords for this analysis
df_with_accords = df.filter(
    F.col("main_accords").isNotNull()
    & (F.trim(F.col("main_accords")) != "[]")
)

# Check the source schema and sample records
df_with_accords.printSchema()
display(df_with_accords.limit(10))

# Validate the filtering results
display(
    df.agg(
        F.count("*").alias("original_fragrances")
    )
    .crossJoin(
        df_with_accords.agg(
            F.count("*").alias("fragrances_with_accords")
        )
    )
    .withColumn(
        "excluded_fragrances",
        F.col("original_fragrances") - F.col("fragrances_with_accords")
    )
)

In [0]:

from pyspark.sql import functions as F

fragrantica = spark.table("main.fragrance_analytics.fragrantica_perfumes")

# Confirm source schema
fragrantica.printSchema()

# Identify records with usable accord data.
# Keep this validation consistent with the existing hierarchy analysis.
# This assumes main_accords is a comma-separated string.
accords_clean = F.trim(F.col("main_accords"))

valid_accords = (
    F.col("main_accords").isNotNull()
    & (accords_clean != "")
    & (~F.lower(accords_clean).isin("null", "none", "nan", "n/a", "[]"))
)

fragrantica_rating_base = fragrantica.withColumn(
    "has_valid_accords",
    valid_accords
)

# Inspect examples with valid accord data
display(
    fragrantica_rating_base
    .filter(F.col("has_valid_accords"))
    .select(
        "id",
        "name",
        "gender",
        "rating_value",
        "rating_count",
        "main_accords"
    )
    .limit(20)
)

# Profile records, distinguishing field presence from valid accord data
fragrantica_rating_base.select(
    F.count("*").alias("total_records"),
    F.countDistinct("id").alias("distinct_ids"),
    F.sum(F.col("rating_value").isNotNull().cast("int"))
        .alias("records_with_rating"),
    F.sum(F.col("rating_count").isNotNull().cast("int"))
        .alias("records_with_rating_count"),
    F.sum(F.col("main_accords").isNotNull().cast("int"))
        .alias("records_with_non_null_accord_field"),
    F.sum(F.col("has_valid_accords").cast("int"))
        .alias("records_with_valid_accord_field"),
    F.sum(
        (F.col("rating_value").isNotNull() & F.col("has_valid_accords"))
        .cast("int")
    ).alias("rated_records_with_valid_accords"),
    F.sum(
        (F.col("rating_value").isNull() & F.col("has_valid_accords"))
        .cast("int")
    ).alias("valid_accord_records_without_rating")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Parse accord arrays while preserving their original order
accord_array = F.from_json(
    F.regexp_replace(F.col("main_accords"), "'", '"'),
    "array<string>"
)

rating_accords_df = (
    fragrantica_rating_base
    .filter(
        F.col("has_valid_accords")
        & F.col("rating_value").isNotNull()
    )
    .withColumn("accords", accord_array)
    .withColumn(
        "accords",
        F.transform("accords", lambda x: F.lower(F.trim(x)))
    )
)

# Extract the first three positions, if present
for position, column_name in [
    (0, "dominant_accord"),
    (1, "second_accord"),
    (2, "third_accord")
]:
    rating_accords_df = rating_accords_df.withColumn(
        column_name,
        F.when(
            (F.size("accords") > position)
            & F.col("accords")[position].isNotNull()
            & (F.trim(F.col("accords")[position]) != ""),
            F.col("accords")[position]
        )
    )

rating_accords_df = rating_accords_df.select(
    "id",
    "name",
    "gender",
    "rating_value",
    "rating_count",
    "dominant_accord",
    "second_accord",
    "third_accord"
)

# Check eligible record counts at each analysis level
rating_accords_df.agg(
    F.count("*").alias("rated_records_with_valid_accords"),
    F.sum(F.col("dominant_accord").isNotNull().cast("int"))
        .alias("eligible_dominant_records"),
    F.sum(
        (
            F.col("dominant_accord").isNotNull()
            & F.col("second_accord").isNotNull()
        ).cast("int")
    ).alias("eligible_pair_records"),
    F.sum(
        (
            F.col("dominant_accord").isNotNull()
            & F.col("second_accord").isNotNull()
            & F.col("third_accord").isNotNull()
        ).cast("int")
    ).alias("eligible_tricord_records")
).show(truncate=False)

display(rating_accords_df.limit(10))

In [0]:
from pyspark.sql import functions as F

# ==========================================
# SECTION 1 — DOMINANT ACCORD RATINGS
# ==========================================

HIGH_RATING_THRESHOLD = 4.5
MIN_ACCORD_FRAGRANCES = 100
MIN_COMMUNITY_VOTES = 100

# Prepare rated fragrances with valid dominant accords
all_rated = rating_accords_df.filter(
    F.col("dominant_accord").isNotNull()
)

# ------------------------------------------
# TABLES 1A & 1B — ALL RATED FRAGRANCES
# ------------------------------------------

overall_rating_baseline = (
    all_rated
    .agg(F.avg("rating_value").alias("overall_mean_rating"))
    .first()["overall_mean_rating"]
)

total_rated_fragrances = all_rated.count()

print(f"Rated fragrances with valid dominant accords: {total_rated_fragrances:,}")
print(f"Overall mean rating: {overall_rating_baseline:.3f}")
print(f"High-rating threshold: {HIGH_RATING_THRESHOLD} or higher")

# Aggregate ratings by dominant accord
dominant_accord_ratings = (
    all_rated
    .groupBy("dominant_accord")
    .agg(
        F.count("*").alias("fragrance_count"),
        F.round(F.avg("rating_value"), 3).alias("mean_rating"),
        F.round(F.median("rating_value"), 3).alias("median_rating"),
        F.sum("rating_count").alias("total_community_ratings"),
        F.sum(
            F.when(
                F.col("rating_value") >= HIGH_RATING_THRESHOLD, 1
            ).otherwise(0)
        ).alias("high_rating_count")
    )
    .withColumn(
        "share_of_rated_dataset_pct",
        F.round(
            F.col("fragrance_count") / F.lit(total_rated_fragrances) * 100,
            2
        )
    )
    .withColumn(
        "high_rating_share_pct",
        F.round(
            F.col("high_rating_count") / F.col("fragrance_count") * 100,
            2
        )
    )
    .withColumn(
        "difference_from_overall_mean",
        F.round(
            F.col("mean_rating") - F.lit(overall_rating_baseline),
            3
        )
    )
)

# Apply minimum group size
dominant_accord_ratings_filtered = (
    dominant_accord_ratings
    .filter(F.col("fragrance_count") >= MIN_ACCORD_FRAGRANCES)
)

print("TABLE 1A — Dominant accords ranked by mean community rating")
display(
    dominant_accord_ratings_filtered
    .orderBy(F.desc("mean_rating"), F.desc("fragrance_count"))
)

print("TABLE 1B — Dominant accords ranked by prevalence")
display(
    dominant_accord_ratings_filtered
    .orderBy(F.desc("fragrance_count"))
)


# ------------------------------------------
# TABLE 1C — SENSITIVITY ANALYSIS
# Individual fragrances must have >= 100 votes
# ------------------------------------------

well_rated = all_rated.filter(
    F.col("rating_count") >= MIN_COMMUNITY_VOTES
)

well_rated_count = well_rated.count()

well_rated_baseline = (
    well_rated
    .agg(F.avg("rating_value").alias("overall_mean_rating"))
    .first()["overall_mean_rating"]
)

print(f"Fragrances with at least {MIN_COMMUNITY_VOTES} community votes: {well_rated_count:,}")
print(f"Overall mean rating for this subset: {well_rated_baseline:.3f}")

dominant_accord_ratings_100_votes = (
    well_rated
    .groupBy("dominant_accord")
    .agg(
        F.count("*").alias("fragrance_count"),
        F.round(F.avg("rating_value"), 3).alias("mean_rating"),
        F.round(F.median("rating_value"), 3).alias("median_rating"),
        F.sum("rating_count").alias("total_community_ratings"),
        F.sum(
            F.when(
                F.col("rating_value") >= HIGH_RATING_THRESHOLD, 1
            ).otherwise(0)
        ).alias("high_rating_count")
    )
    .withColumn(
        "share_of_filtered_dataset_pct",
        F.round(
            F.col("fragrance_count") / F.lit(well_rated_count) * 100,
            2
        )
    )
    .withColumn(
        "high_rating_share_pct",
        F.round(
            F.col("high_rating_count") / F.col("fragrance_count") * 100,
            2
        )
    )
    .withColumn(
        "difference_from_filtered_mean",
        F.round(
            F.col("mean_rating") - F.lit(well_rated_baseline),
            3
        )
    )
    .filter(F.col("fragrance_count") >= MIN_ACCORD_FRAGRANCES)
)

print(
    f"TABLE 1C — Dominant accord ratings; each fragrance has at least "
    f"{MIN_COMMUNITY_VOTES} community votes"
)
display(
    dominant_accord_ratings_100_votes
    .orderBy(F.desc("mean_rating"), F.desc("fragrance_count"))
)


# ------------------------------------------
# TABLE 1D — ORIGINAL VS FILTERED COMPARISON
# ------------------------------------------

original_comparison = (
    dominant_accord_ratings_filtered
    .select(
        "dominant_accord",
        F.col("fragrance_count").alias("original_fragrance_count"),
        F.col("mean_rating").alias("original_mean_rating"),
        F.col("median_rating").alias("original_median_rating"),
        F.col("share_of_rated_dataset_pct").alias("original_share_pct"),
        F.col("high_rating_count").alias("original_high_rating_count"),
        F.col("high_rating_share_pct").alias("original_high_rating_share_pct"),
        F.col("difference_from_overall_mean").alias(
            "original_difference_from_mean"
        )
    )
)

filtered_comparison = (
    dominant_accord_ratings_100_votes
    .select(
        "dominant_accord",
        F.col("fragrance_count").alias("filtered_fragrance_count"),
        F.col("mean_rating").alias("filtered_mean_rating"),
        F.col("median_rating").alias("filtered_median_rating"),
        F.col("share_of_filtered_dataset_pct").alias("filtered_share_pct"),
        F.col("high_rating_count").alias("filtered_high_rating_count"),
        F.col("high_rating_share_pct").alias("filtered_high_rating_share_pct"),
        F.col("difference_from_filtered_mean").alias(
            "filtered_difference_from_mean"
        )
    )
)

dominant_accord_comparison = (
    original_comparison
    .join(
        filtered_comparison,
        on="dominant_accord",
        how="full_outer"
    )
    .withColumn(
        "change_in_mean_rating",
        F.round(
            F.col("filtered_mean_rating") - F.col("original_mean_rating"),
            3
        )
    )
    .withColumn(
        "change_in_mean_rating_pct",
        F.round(
            (
                F.col("filtered_mean_rating") - F.col("original_mean_rating")
            ) / F.col("original_mean_rating") * 100,
            2
        )
    )
    .withColumn(
        "change_in_high_rating_share_pp",
        F.round(
            F.col("filtered_high_rating_share_pct")
            - F.col("original_high_rating_share_pct"),
            2
        )
    )
    .withColumn(
        "change_in_fragrance_count",
        F.col("filtered_fragrance_count") - F.col("original_fragrance_count")
    )
    .withColumn(
        "filtered_fragrance_retention_pct",
        F.round(
            F.col("filtered_fragrance_count")
            / F.col("original_fragrance_count") * 100,
            1
        )
    )
    .select(
        "dominant_accord",
        "original_fragrance_count",
        "filtered_fragrance_count",
        "filtered_fragrance_retention_pct",
        "change_in_fragrance_count",
        "original_mean_rating",
        "filtered_mean_rating",
        "change_in_mean_rating",
        "change_in_mean_rating_pct",
        "original_median_rating",
        "filtered_median_rating",
        "original_share_pct",
        "filtered_share_pct",
        "original_difference_from_mean",
        "filtered_difference_from_mean",
        "original_high_rating_count",
        "filtered_high_rating_count",
        "original_high_rating_share_pct",
        "filtered_high_rating_share_pct",
        "change_in_high_rating_share_pp"
    )
)

print(
    "TABLE 1D — Original vs. 100-vote sensitivity analysis "
    "by dominant accord"
)
display(
    dominant_accord_comparison
    .orderBy(F.desc("change_in_mean_rating"))
)

In [0]:

from pyspark.sql import functions as F
from scipy.stats import t

# ==========================================
# SECTION 1 — CONFIDENCE INTERVALS
# ==========================================

HIGH_RATING_THRESHOLD = 4.5
CONFIDENCE_LEVEL = 0.95
ALPHA = 1 - CONFIDENCE_LEVEL
Z_WILSON = 1.96  # 95% Wilson interval for proportions

# Use the same population as the descriptive baseline
ci_base = rating_accords_df.filter(
    F.col("dominant_accord").isNotNull()
    & F.col("rating_value").isNotNull()
)

# ------------------------------------------
# 1. AGGREGATE STATISTICS BY DOMINANT ACCORD
# ------------------------------------------

accord_ci = (
    ci_base
    .groupBy("dominant_accord")
    .agg(
        F.count("*").alias("fragrance_count"),
        F.avg("rating_value").alias("mean_rating"),
        F.stddev_samp("rating_value").alias("rating_stddev"),
        F.sum(
            F.when(
                F.col("rating_value") >= HIGH_RATING_THRESHOLD, 1
            ).otherwise(0)
        ).alias("high_rating_count")
    )
    .withColumn(
        "high_rating_share",
        F.col("high_rating_count") / F.col("fragrance_count")
    )
)

# ------------------------------------------
# 2. STUDENT'S t CRITICAL VALUES FOR MEANS
# ------------------------------------------
# Calculate one t critical value per accord sample size.
# Only the small set of aggregate sample sizes is collected.

sample_sizes = [
    row["fragrance_count"]
    for row in accord_ci.select("fragrance_count").distinct().collect()
]

t_critical_rows = [
    (int(n), float(t.ppf(1 - ALPHA / 2, df=int(n) - 1)))
    for n in sample_sizes
    if n > 1
]

t_critical_df = spark.createDataFrame(
    t_critical_rows,
    ["fragrance_count", "t_critical"]
)

accord_ci = (
    accord_ci
    .join(t_critical_df, on="fragrance_count", how="left")
    .withColumn(
        "mean_standard_error",
        F.col("rating_stddev") / F.sqrt(F.col("fragrance_count"))
    )
    .withColumn(
        "mean_ci_lower",
        F.col("mean_rating")
        - F.col("t_critical") * F.col("mean_standard_error")
    )
    .withColumn(
        "mean_ci_upper",
        F.col("mean_rating")
        + F.col("t_critical") * F.col("mean_standard_error")
    )
    .withColumn(
        "mean_ci_width",
        F.col("mean_ci_upper") - F.col("mean_ci_lower")
    )
)

# ------------------------------------------
# 3. WILSON 95% CI FOR HIGH-RATING SHARE
# ------------------------------------------

z = F.lit(Z_WILSON)
n = F.col("fragrance_count").cast("double")
p = F.col("high_rating_share")

denominator = 1 + (z * z / n)

wilson_center = (
    p + z * z / (2 * n)
) / denominator

wilson_half_width = (
    z * F.sqrt(
        (p * (1 - p) / n)
        + (z * z / (4 * n * n))
    ) / denominator
)

accord_ci = (
    accord_ci
    .withColumn(
        "high_share_ci_lower_pct",
        F.round((wilson_center - wilson_half_width) * 100, 2)
    )
    .withColumn(
        "high_share_ci_upper_pct",
        F.round((wilson_center + wilson_half_width) * 100, 2)
    )
    .withColumn(
        "high_rating_share_pct",
        F.round(p * 100, 2)
    )
    .withColumn(
        "mean_rating",
        F.round("mean_rating", 3)
    )
    .withColumn(
        "mean_ci_lower",
        F.round("mean_ci_lower", 3)
    )
    .withColumn(
        "mean_ci_upper",
        F.round("mean_ci_upper", 3)
    )
    .withColumn(
        "mean_ci_width",
        F.round("mean_ci_width", 3)
    )
    .withColumn(
        "rating_stddev",
        F.round("rating_stddev", 3)
    )
)

# ------------------------------------------
# 4. DISPLAY RESULTS
# ------------------------------------------

print(
    "95% confidence intervals by dominant accord "
    "(Student's t for means; Wilson for proportions)"
)

display(
    accord_ci
    .select(
        "dominant_accord",
        "fragrance_count",
        "mean_rating",
        "rating_stddev",
        "mean_ci_lower",
        "mean_ci_upper",
        "mean_ci_width",
        "high_rating_count",
        "high_rating_share_pct",
        "high_share_ci_lower_pct",
        "high_share_ci_upper_pct"
    )
    .orderBy(F.desc("fragrance_count"))
)

In [0]:
### Section 2 — Pairwise Statistical Comparisons

from pyspark.sql import functions as F
from scipy.stats import t
from itertools import combinations
import pandas as pd

# ==========================================
# SECTION 2 — PAIRWISE STATISTICAL COMPARISONS
# ==========================================

CONFIDENCE_LEVEL = 0.95
ALPHA = 1 - CONFIDENCE_LEVEL
MIN_ACCORD_FRAGRANCES = 100

# ------------------------------------------
# 1. AGGREGATE UNROUNDED STATISTICS
# ------------------------------------------
# Use the original rated-fragrance population.
# Do not reuse rounded statistics from Section 1 or 2.

comparison_base = rating_accords_df.filter(
    F.col("dominant_accord").isNotNull()
    & F.col("rating_value").isNotNull()
)

accord_stats_df = (
    comparison_base
    .groupBy("dominant_accord")
    .agg(
        F.count("*").alias("fragrance_count"),
        F.avg("rating_value").alias("mean_rating"),
        F.stddev_samp("rating_value").alias("rating_stddev")
    )
    .filter(F.col("fragrance_count") >= MIN_ACCORD_FRAGRANCES)
)

# Collect only the small aggregate table, not individual fragrances.
accord_stats = [
    {
        "accord": row["dominant_accord"],
        "n": int(row["fragrance_count"]),
        "mean": float(row["mean_rating"]),
        "sd": float(row["rating_stddev"])
    }
    for row in accord_stats_df.collect()
]

# Sort by observed mean rating, highest first.
accord_stats = sorted(
    accord_stats,
    key=lambda x: x["mean"],
    reverse=True
)

print(f"Accord groups included: {len(accord_stats)}")
print(f"Minimum fragrances per group: {MIN_ACCORD_FRAGRANCES}")

# ------------------------------------------
# 2. WELCH'S T-TEST FOR EVERY PAIR
# ------------------------------------------
# Difference = mean rating of the higher-rated group
#              minus mean rating of the lower-rated group.
#
# Welch's test does not assume equal group variances.
# The confidence intervals below are individual, unadjusted
# 95% intervals. Holm-adjusted p-values control for multiple tests.

pairwise_results = []

for group_a, group_b in combinations(accord_stats, 2):

    n_a, n_b = group_a["n"], group_b["n"]
    mean_a, mean_b = group_a["mean"], group_b["mean"]
    sd_a, sd_b = group_a["sd"], group_b["sd"]

    difference = mean_a - mean_b

    variance_component_a = (sd_a ** 2) / n_a
    variance_component_b = (sd_b ** 2) / n_b

    standard_error = (
        variance_component_a + variance_component_b
    ) ** 0.5

    # Welch-Satterthwaite degrees of freedom
    denominator = (
        (variance_component_a ** 2) / (n_a - 1)
        + (variance_component_b ** 2) / (n_b - 1)
    )

    if standard_error == 0 or denominator == 0:
        continue

    degrees_freedom = (
        (variance_component_a + variance_component_b) ** 2
        / denominator
    )

    t_statistic = difference / standard_error

    p_value = float(
        2 * t.sf(abs(t_statistic), df=degrees_freedom)
    )

    t_critical = float(
        t.ppf(1 - ALPHA / 2, df=degrees_freedom)
    )

    ci_lower = difference - t_critical * standard_error
    ci_upper = difference + t_critical * standard_error

    # Approximate standardized effect size:
    # Cohen's d using the pooled within-group standard deviation.
    pooled_sd = (
        (
            (n_a - 1) * sd_a ** 2
            + (n_b - 1) * sd_b ** 2
        ) / (n_a + n_b - 2)
    ) ** 0.5

    cohens_d = (
        difference / pooled_sd
        if pooled_sd > 0
        else None
    )

    pairwise_results.append({
        "higher_rated_accord": group_a["accord"],
        "lower_rated_accord": group_b["accord"],
        "higher_group_n": n_a,
        "lower_group_n": n_b,
        "higher_group_mean": mean_a,
        "lower_group_mean": mean_b,
        "mean_difference": difference,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "welch_t": t_statistic,
        "degrees_freedom": degrees_freedom,
        "p_value": p_value,
        "cohens_d": cohens_d
    })

# ------------------------------------------
# 3. HOLM MULTIPLE-COMPARISON CORRECTION
# ------------------------------------------

number_of_tests = len(pairwise_results)

if number_of_tests == 0:
    raise ValueError(
        "No valid pairwise comparisons were produced. "
        "Check the group-size threshold and source data."
    )

# Sort p-values from smallest to largest.
sorted_indices = sorted(
    range(number_of_tests),
    key=lambda i: pairwise_results[i]["p_value"]
)

# Holm step-down adjustment with monotonic adjusted p-values.
running_max = 0.0

for rank, index in enumerate(sorted_indices):
    raw_p = pairwise_results[index]["p_value"]

    adjusted_p = min(
        1.0,
        (number_of_tests - rank) * raw_p
    )

    running_max = max(running_max, adjusted_p)

    pairwise_results[index]["holm_adjusted_p"] = running_max

# ------------------------------------------
# 4. PREPARE THE RESULTS TABLE
# ------------------------------------------

results_df = pd.DataFrame(pairwise_results)

results_df["significant_after_holm"] = (
    results_df["holm_adjusted_p"] < ALPHA
)

# Round only for display, after all calculations.
display_columns = [
    "higher_rated_accord",
    "lower_rated_accord",
    "higher_group_n",
    "lower_group_n",
    "higher_group_mean",
    "lower_group_mean",
    "mean_difference",
    "ci_lower",
    "ci_upper",
    "welch_t",
    "degrees_freedom",
    "p_value",
    "holm_adjusted_p",
    "cohens_d",
    "significant_after_holm"
]

results_display = results_df[display_columns].copy()

round_columns = [
    "higher_group_mean",
    "lower_group_mean",
    "mean_difference",
    "ci_lower",
    "ci_upper",
    "welch_t",
    "degrees_freedom",
    "p_value",
    "holm_adjusted_p",
    "cohens_d"
]

results_display[round_columns] = (
    results_display[round_columns].round(4)
)

print(f"Pairwise tests performed: {number_of_tests:,}")
print(f"Confidence level: {CONFIDENCE_LEVEL:.0%}")
print("Multiple-testing correction: Holm")
print(
    "Significant comparisons after correction: "
    f"{int(results_df['significant_after_holm'].sum()):,}"
)

# Show the strongest statistically supported differences first.
display(
    results_display
    .sort_values(
        ["holm_adjusted_p", "mean_difference"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

In [0]:
### Section 3 — Sensitivity Analysis: Fragrances with at Least 100 Community Votes

from pyspark.sql import functions as F
from scipy.stats import t
from itertools import combinations
import pandas as pd

# ==========================================
# SECTION 3 — SENSITIVITY ANALYSIS
# ==========================================

MIN_COMMUNITY_VOTES = 100
MIN_ACCORD_FRAGRANCES = 100
ALPHA = 0.05

# ------------------------------------------
# 1. FILTER AND AGGREGATE
# ------------------------------------------

sensitivity_base = rating_accords_df.filter(
    F.col("dominant_accord").isNotNull()
    & F.col("rating_value").isNotNull()
    & F.col("rating_count").isNotNull()
    & (F.col("rating_count") >= MIN_COMMUNITY_VOTES)
)

sensitivity_stats_df = (
    sensitivity_base
    .groupBy("dominant_accord")
    .agg(
        F.count("*").alias("fragrance_count"),
        F.avg("rating_value").alias("mean_rating"),
        F.stddev_samp("rating_value").alias("rating_stddev")
    )
    .filter(F.col("fragrance_count") >= MIN_ACCORD_FRAGRANCES)
)

sensitivity_stats = [
    {
        "accord": row["dominant_accord"],
        "n": int(row["fragrance_count"]),
        "mean": float(row["mean_rating"]),
        "sd": float(row["rating_stddev"])
    }
    for row in sensitivity_stats_df.collect()
    if row["rating_stddev"] is not None
]

print(f"Vote threshold per fragrance: {MIN_COMMUNITY_VOTES}")
print(f"Accord groups included: {len(sensitivity_stats)}")

# ------------------------------------------
# 2. WELCH PAIRWISE TESTS
# ------------------------------------------

sensitivity_results = []

for group_a, group_b in combinations(sensitivity_stats, 2):
    n_a, n_b = group_a["n"], group_b["n"]
    mean_a, mean_b = group_a["mean"], group_b["mean"]
    sd_a, sd_b = group_a["sd"], group_b["sd"]

    difference = mean_a - mean_b
    var_a = sd_a ** 2 / n_a
    var_b = sd_b ** 2 / n_b
    se = (var_a + var_b) ** 0.5

    denominator = (
        var_a ** 2 / (n_a - 1)
        + var_b ** 2 / (n_b - 1)
    )

    if se == 0 or denominator == 0:
        continue

    df_welch = (var_a + var_b) ** 2 / denominator
    t_stat = difference / se
    p_value = float(2 * t.sf(abs(t_stat), df=df_welch))
    t_critical = float(t.ppf(1 - ALPHA / 2, df=df_welch))

    pooled_sd = (
        ((n_a - 1) * sd_a ** 2 + (n_b - 1) * sd_b ** 2)
        / (n_a + n_b - 2)
    ) ** 0.5

    sensitivity_results.append({
        "higher_rated_accord": group_a["accord"],
        "lower_rated_accord": group_b["accord"],
        "higher_group_n": n_a,
        "lower_group_n": n_b,
        "higher_group_mean": mean_a,
        "lower_group_mean": mean_b,
        "mean_difference": difference,
        "ci_lower": difference - t_critical * se,
        "ci_upper": difference + t_critical * se,
        "p_value": p_value,
        "cohens_d": difference / pooled_sd if pooled_sd > 0 else None
    })

if not sensitivity_results:
    raise ValueError("No valid pairwise comparisons were produced.")

# ------------------------------------------
# 3. HOLM CORRECTION
# ------------------------------------------

sorted_indices = sorted(
    range(len(sensitivity_results)),
    key=lambda i: sensitivity_results[i]["p_value"]
)

running_max = 0.0
number_of_tests = len(sensitivity_results)

for rank, index in enumerate(sorted_indices):
    adjusted_p = min(
        1.0,
        (number_of_tests - rank)
        * sensitivity_results[index]["p_value"]
    )
    running_max = max(running_max, adjusted_p)
    sensitivity_results[index]["holm_adjusted_p"] = running_max

sensitivity_df = pd.DataFrame(sensitivity_results)
sensitivity_df["significant_after_holm"] = (
    sensitivity_df["holm_adjusted_p"] < ALPHA
)

print(f"Pairwise tests performed: {number_of_tests:,}")
print(
    "Significant comparisons after Holm correction: "
    f"{int(sensitivity_df['significant_after_holm'].sum()):,}"
)

# ------------------------------------------
# 4. COMPARE WITH ORIGINAL RESULTS
# ------------------------------------------
# results_df comes from Section 3.
# Match comparisons by their two accord names.

original_for_merge = results_df[
    [
        "higher_rated_accord",
        "lower_rated_accord",
        "mean_difference",
        "holm_adjusted_p",
        "significant_after_holm"
    ]
].rename(columns={
    "mean_difference": "original_mean_difference",
    "holm_adjusted_p": "original_holm_p",
    "significant_after_holm": "original_significant"
})

sensitivity_for_merge = sensitivity_df[
    [
        "higher_rated_accord",
        "lower_rated_accord",
        "higher_group_n",
        "lower_group_n",
        "mean_difference",
        "ci_lower",
        "ci_upper",
        "holm_adjusted_p",
        "significant_after_holm",
        "cohens_d"
    ]
].rename(columns={
    "mean_difference": "filtered_mean_difference",
    "ci_lower": "filtered_ci_lower",
    "ci_upper": "filtered_ci_upper",
    "holm_adjusted_p": "filtered_holm_p",
    "significant_after_holm": "filtered_significant",
    "cohens_d": "filtered_cohens_d"
})

sensitivity_comparison = original_for_merge.merge(
    sensitivity_for_merge,
    on=["higher_rated_accord", "lower_rated_accord"],
    how="inner"
)

sensitivity_comparison["change_in_difference"] = (
    sensitivity_comparison["filtered_mean_difference"]
    - sensitivity_comparison["original_mean_difference"]
)

sensitivity_comparison["significance_changed"] = (
    sensitivity_comparison["original_significant"]
    != sensitivity_comparison["filtered_significant"]
)

print(
    "Comparisons significant in both analyses: "
    f"{int((sensitivity_comparison['original_significant'] & sensitivity_comparison['filtered_significant']).sum()):,}"
)
print(
    "Comparisons losing significance after filtering: "
    f"{int((sensitivity_comparison['original_significant'] & ~sensitivity_comparison['filtered_significant']).sum()):,}"
)
print(
    "Comparisons gaining significance after filtering: "
    f"{int((~sensitivity_comparison['original_significant'] & sensitivity_comparison['filtered_significant']).sum()):,}"
)

# Show the original strongest differences and their filtered equivalents.
display(
    sensitivity_comparison
    .sort_values("original_mean_difference", ascending=False)
    [[
        "higher_rated_accord",
        "lower_rated_accord",
        "original_mean_difference",
        "filtered_mean_difference",
        "change_in_difference",
        "filtered_ci_lower",
        "filtered_ci_upper",
        "original_holm_p",
        "filtered_holm_p",
        "original_significant",
        "filtered_significant",
        "significance_changed"
    ]]
    .head(20)
    .round(4)
)

In [0]:

# ==========================================
# Visualization 1: Mean rating by dominant accord
# ==========================================

import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import t
from pyspark.sql import functions as F

# ------------------------------------------
# Configuration
# ------------------------------------------

MIN_ACCORD_FRAGRANCES = 100
ALPHA = 0.05

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)
OUTPUT_FILENAME = "dominant-accord-mean-ratings.png"
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

# ------------------------------------------
# Aggregate ratings
# ------------------------------------------

ratings_summary = (
    rating_accords_df
    .filter(
        F.col("dominant_accord").isNotNull()
        & F.col("rating_value").isNotNull()
    )
    .groupBy("dominant_accord")
    .agg(
        F.count("*").alias("fragrance_count"),
        F.avg("rating_value").alias("mean_rating"),
        F.stddev_samp("rating_value").alias("rating_stddev")
    )
    .filter(F.col("fragrance_count") >= MIN_ACCORD_FRAGRANCES)
    .toPandas()
)

if ratings_summary.empty:
    raise ValueError("No accord groups meet the minimum sample-size threshold.")

# ------------------------------------------
# Calculate 95% t-confidence intervals
# ------------------------------------------

ratings_summary["standard_error"] = (
    ratings_summary["rating_stddev"]
    / np.sqrt(ratings_summary["fragrance_count"])
)

ratings_summary["t_critical"] = ratings_summary["fragrance_count"].apply(
    lambda n: t.ppf(1 - ALPHA / 2, df=n - 1)
)

ratings_summary["ci_lower"] = (
    ratings_summary["mean_rating"]
    - ratings_summary["t_critical"] * ratings_summary["standard_error"]
)

ratings_summary["ci_upper"] = (
    ratings_summary["mean_rating"]
    + ratings_summary["t_critical"] * ratings_summary["standard_error"]
)

# Order from lowest to highest so the highest appears at the top
chart_df = (
    ratings_summary
    .sort_values("mean_rating", ascending=True)
    .reset_index(drop=True)
)

chart_df["accord_label"] = chart_df["dominant_accord"].str.title()

# ------------------------------------------
# Palette — matching previous visualizations
# ------------------------------------------

background = "#21171f"
text_color = "#f4ece9"
muted_color = "#c6b6c0"
gold = "#d6b47a"
bar_color = "#98765b"
border_color = "#4d3948"

# ------------------------------------------
# Plot
# ------------------------------------------

fig, ax = plt.subplots(figsize=(12, 9), dpi=180)

fig.patch.set_facecolor(background)
ax.set_facecolor(background)

y_positions = np.arange(len(chart_df))

point_colors = [
    gold if i == len(chart_df) - 1 else bar_color
    for i in range(len(chart_df))
]

for i, row in chart_df.iterrows():
    color = point_colors[i]

    ax.errorbar(
        x=row["mean_rating"],
        y=y_positions[i],
        xerr=[[
            row["mean_rating"] - row["ci_lower"]
        ], [
            row["ci_upper"] - row["mean_rating"]
        ]],
        fmt="o",
        color=color,
        ecolor=color,
        markersize=6,
        elinewidth=1.5,
        capsize=3,
        markeredgewidth=0,
        zorder=3
    )

    ax.text(
        row["ci_upper"] + 0.012,
        y_positions[i],
        f'{row["mean_rating"]:.3f}  (n={row["fragrance_count"]:,})',
        va="center",
        ha="left",
        color=text_color,
        fontsize=8
    )

fig.text(
    0.13, 0.965,
    "FRAGRANCE INTELLIGENCE  /  RATINGS ANALYSIS",
    color=gold,
    fontsize=9,
    fontweight="bold",
    ha="left"
)

fig.text(
    0.13, 0.925,
    "Which accords receive higher ratings?",
    color=text_color,
    fontsize=22,
    fontfamily="serif",
    ha="left"
)

fig.text(
    0.13, 0.895,
    "Mean community rating by dominant accord · 95% confidence intervals",
    color=muted_color,
    fontsize=10,
    ha="left"
)

ax.set_yticks(y_positions)
ax.set_yticklabels(chart_df["accord_label"])

ax.set_xlabel(
    "Mean community rating (0–5 scale)",
    color=muted_color,
    fontsize=10,
    labelpad=12
)
ax.set_ylabel("")

# Focus on the observed range for readability.
# The caption explicitly identifies the full 0–5 rating scale.
x_min = max(
    0,
    float(chart_df["ci_lower"].min()) - 0.12
)
x_max = float(chart_df["ci_upper"].max()) + 0.20
ax.set_xlim(x_min, x_max)

ax.tick_params(
    axis="y", colors=text_color, labelsize=9, length=0
)
ax.tick_params(
    axis="x", colors=muted_color, labelsize=9, length=0, pad=8
)

ax.grid(
    axis="x",
    color=border_color,
    linewidth=0.8,
    alpha=0.7,
    zorder=0
)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

fig.text(
    0.5, 0.025,
    "Groups with at least 100 fragrances. Whiskers show individual 95% confidence intervals; n is the number of fragrance records.",
    color=muted_color,
    fontsize=7.5,
    ha="center"
)

plt.subplots_adjust(
    left=0.23,
    right=0.91,
    top=0.84,
    bottom=0.10
)

# ------------------------------------------
# Save PNG to Databricks Volume
# ------------------------------------------

if not os.path.isdir(OUTPUT_DIR):
    raise FileNotFoundError(
        f"Visualization directory does not exist: {OUTPUT_DIR}"
    )

fig.savefig(
    output_path,
    dpi=180,
    facecolor=fig.get_facecolor(),
    bbox_inches=None
)

if not os.path.isfile(output_path):
    raise RuntimeError(f"Chart was not created: {output_path}")

print(f"Chart saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path):,} bytes")
print(f"PNG dimensions: {fig.canvas.get_width_height()}")
print(f"Accord groups plotted: {len(chart_df)}")

plt.show()
plt.close(fig)


In [0]:

# ==========================================
# Visualization 2: Accord ratings vs overall mean
# ==========================================

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t
from pyspark.sql import functions as F

# ------------------------------------------
# Configuration
# ------------------------------------------

MIN_ACCORD_FRAGRANCES = 100
ALPHA = 0.05

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)
OUTPUT_FILENAME = "dominant-accord-vs-overall-mean.png"
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

# ------------------------------------------
# Define analysis population and baseline
# ------------------------------------------

analysis_df = rating_accords_df.filter(
    F.col("dominant_accord").isNotNull()
    & F.col("rating_value").isNotNull()
)

overall_mean = analysis_df.agg(
    F.avg("rating_value").alias("overall_mean")
).first()["overall_mean"]

# ------------------------------------------
# Calculate accord-level summary statistics
# ------------------------------------------

ratings_summary = (
    analysis_df
    .groupBy("dominant_accord")
    .agg(
        F.count("*").alias("fragrance_count"),
        F.avg("rating_value").alias("mean_rating"),
        F.stddev_samp("rating_value").alias("rating_stddev")
    )
    .filter(F.col("fragrance_count") >= MIN_ACCORD_FRAGRANCES)
    .toPandas()
)

if ratings_summary.empty:
    raise ValueError("No accord groups meet the minimum sample-size threshold.")

# ------------------------------------------
# Calculate individual 95% confidence intervals
# ------------------------------------------

ratings_summary["standard_error"] = (
    ratings_summary["rating_stddev"]
    / np.sqrt(ratings_summary["fragrance_count"])
)

ratings_summary["t_critical"] = ratings_summary["fragrance_count"].apply(
    lambda n: t.ppf(1 - ALPHA / 2, df=n - 1)
)

ratings_summary["ci_lower"] = (
    ratings_summary["mean_rating"]
    - ratings_summary["t_critical"] * ratings_summary["standard_error"]
)

ratings_summary["ci_upper"] = (
    ratings_summary["mean_rating"]
    + ratings_summary["t_critical"] * ratings_summary["standard_error"]
)

ratings_summary["difference_from_overall"] = (
    ratings_summary["mean_rating"] - overall_mean
)

# Order by difference from the overall mean
chart_df = (
    ratings_summary
    .sort_values("difference_from_overall", ascending=True)
    .reset_index(drop=True)
)

chart_df["accord_label"] = chart_df["dominant_accord"].str.title()

# ------------------------------------------
# Palette — matching previous visualizations
# ------------------------------------------

background = "#21171f"
text_color = "#f4ece9"
muted_color = "#c6b6c0"
gold = "#d6b47a"
bar_color = "#98765b"
border_color = "#4d3948"

# ------------------------------------------
# Plot
# ------------------------------------------

fig, ax = plt.subplots(figsize=(12, 9), dpi=180)

fig.patch.set_facecolor(background)
ax.set_facecolor(background)

y_positions = np.arange(len(chart_df))

for i, row in chart_df.iterrows():
    mean_rating = row["mean_rating"]
    color = gold if mean_rating >= overall_mean else bar_color

    # Plot mean rating and its individual confidence interval
    ax.errorbar(
        x=mean_rating,
        y=y_positions[i],
        xerr=[[
            mean_rating - row["ci_lower"]
        ], [
            row["ci_upper"] - mean_rating
        ]],
        fmt="o",
        color=color,
        ecolor=color,
        markersize=6,
        elinewidth=1.5,
        capsize=3,
        markeredgewidth=0,
        zorder=3
    )

    # Label with mean, difference from baseline, and sample size
    difference = row["difference_from_overall"]
    sign = "+" if difference >= 0 else ""

    ax.text(
        row["ci_upper"] + 0.012,
        y_positions[i],
        f'{mean_rating:.3f}  ({sign}{difference:.3f}; n={row["fragrance_count"]:,})',
        va="center",
        ha="left",
        color=text_color,
        fontsize=7.5
    )

# Overall mean reference line
ax.axvline(
    overall_mean,
    color=muted_color,
    linewidth=1.2,
    linestyle="--",
    alpha=0.9,
    zorder=1
)

fig.text(
    0.13, 0.965,
    "FRAGRANCE INTELLIGENCE  /  RATINGS ANALYSIS",
    color=gold,
    fontsize=9,
    fontweight="bold",
    ha="left"
)

fig.text(
    0.13, 0.925,
    "Which accords stand above the average?",
    color=text_color,
    fontsize=21,
    fontfamily="serif",
    ha="left"
)

fig.text(
    0.13, 0.895,
    f"Mean community rating by dominant accord · Overall mean = {overall_mean:.3f}",
    color=muted_color,
    fontsize=10,
    ha="left"
)

ax.set_yticks(y_positions)
ax.set_yticklabels(chart_df["accord_label"])

ax.set_xlabel(
    "Mean community rating (0–5 scale)",
    color=muted_color,
    fontsize=10,
    labelpad=12
)
ax.set_ylabel("")

# Focus on the observed ratings range
x_min = max(0, float(chart_df["ci_lower"].min()) - 0.12)
x_max = float(chart_df["ci_upper"].max()) + 0.30
ax.set_xlim(x_min, x_max)

ax.tick_params(
    axis="y", colors=text_color, labelsize=9, length=0
)
ax.tick_params(
    axis="x", colors=muted_color, labelsize=9, length=0, pad=8
)

ax.grid(
    axis="x",
    color=border_color,
    linewidth=0.8,
    alpha=0.7,
    zorder=0
)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

fig.text(
    0.5, 0.025,
    "Groups with at least 100 fragrances. Dashed line = overall mean. "
    "Whiskers = individual 95% confidence intervals. Labels show mean, "
    "difference from overall mean, and sample size.",
    color=muted_color,
    fontsize=7,
    ha="center"
)

plt.subplots_adjust(
    left=0.23,
    right=0.91,
    top=0.84,
    bottom=0.10
)

# ------------------------------------------
# Save PNG to Databricks Volume
# ------------------------------------------

if not os.path.isdir(OUTPUT_DIR):
    raise FileNotFoundError(
        f"Visualization directory does not exist: {OUTPUT_DIR}"
    )

fig.savefig(
    output_path,
    dpi=180,
    facecolor=fig.get_facecolor()
)

if not os.path.isfile(output_path):
    raise RuntimeError(f"Chart was not created: {output_path}")

print(f"Chart saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path):,} bytes")
print(f"PNG dimensions: {fig.canvas.get_width_height()}")
print(f"Overall mean rating: {overall_mean:.4f}")
print(f"Accord groups plotted: {len(chart_df)}")

plt.show()
plt.close(fig)


In [0]:

# ==========================================
# Visualization 3: Community votes vs rating
# ==========================================

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from pyspark.sql import functions as F

# ------------------------------------------
# Configuration
# ------------------------------------------

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)
OUTPUT_FILENAME = "fragrance-votes-vs-ratings.png"
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

# ------------------------------------------
# Prepare one row per rated fragrance
# ------------------------------------------

scatter_df = (
    rating_accords_df
    .filter(
        F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") > 0)
    )
    .select("id", "name", "rating_value", "rating_count")
    .dropDuplicates(["id"])
    .toPandas()
)

if scatter_df.empty:
    raise ValueError("No fragrances with positive vote counts and valid ratings.")

scatter_df["rating_value"] = scatter_df["rating_value"].astype(float)
scatter_df["rating_count"] = scatter_df["rating_count"].astype(float)

# Keep valid values for plotting and correlation.
scatter_df = scatter_df[
    np.isfinite(scatter_df["rating_value"])
    & np.isfinite(scatter_df["rating_count"])
    & (scatter_df["rating_count"] > 0)
].copy()

if scatter_df.empty:
    raise ValueError("No valid observations remain after filtering.")

# ------------------------------------------
# Calculate Spearman correlation
# ------------------------------------------

rho, p_value = spearmanr(
    scatter_df["rating_count"],
    scatter_df["rating_value"]
)

# ------------------------------------------
# Calculate median ratings in log-spaced bins
# ------------------------------------------
# The trend line summarizes the median rating at different
# vote-count ranges. It is descriptive, not a fitted causal model.

min_votes = scatter_df["rating_count"].min()
max_votes = scatter_df["rating_count"].max()

if min_votes == max_votes:
    raise ValueError("All fragrances have the same vote count; log bins cannot be formed.")

bin_edges = np.geomspace(min_votes, max_votes * (1 + 1e-9), 25)

scatter_df["vote_bin"] = pd.cut(
    scatter_df["rating_count"],
    bins=np.unique(bin_edges),
    include_lowest=True,
    duplicates="drop"
)

trend_df = (
    scatter_df
    .groupby("vote_bin", observed=True)
    .agg(
        median_votes=("rating_count", "median"),
        median_rating=("rating_value", "median"),
        fragrance_count=("rating_value", "size")
    )
    .dropna()
    .sort_values("median_votes")
)

# ------------------------------------------
# Palette — matching previous visualizations
# ------------------------------------------

background = "#21171f"
text_color = "#f4ece9"
muted_color = "#c6b6c0"
gold = "#d6b47a"
bar_color = "#98765b"
border_color = "#4d3948"

# ------------------------------------------
# Plot
# ------------------------------------------

fig, ax = plt.subplots(figsize=(12, 9), dpi=180)

fig.patch.set_facecolor(background)
ax.set_facecolor(background)

ax.scatter(
    scatter_df["rating_count"],
    scatter_df["rating_value"],
    s=8,
    color=bar_color,
    alpha=0.18,
    edgecolors="none",
    rasterized=True,
    zorder=2
)

# Median trend line across logarithmic vote-count bins
if not trend_df.empty:
    ax.plot(
        trend_df["median_votes"],
        trend_df["median_rating"],
        color=gold,
        linewidth=2.2,
        marker="o",
        markersize=4,
        label="Median rating by vote-count range",
        zorder=4
    )

ax.set_xscale("log")

fig.text(
    0.13, 0.965,
    "FRAGRANCE INTELLIGENCE  /  RATINGS ANALYSIS",
    color=gold,
    fontsize=9,
    fontweight="bold",
    ha="left"
)

fig.text(
    0.13, 0.925,
    "Do more-voted fragrances receive different ratings?",
    color=text_color,
    fontsize=20,
    fontfamily="serif",
    ha="left"
)

fig.text(
    0.13, 0.895,
    f"{len(scatter_df):,} fragrances · Spearman correlation = {rho:.3f}",
    color=muted_color,
    fontsize=10,
    ha="left"
)

ax.set_xlabel(
    "Number of community votes (log scale)",
    color=muted_color,
    fontsize=10,
    labelpad=12
)
ax.set_ylabel(
    "Average community rating",
    color=muted_color,
    fontsize=10,
    labelpad=12
)

ax.tick_params(axis="x", colors=muted_color, labelsize=9, length=0, pad=8)
ax.tick_params(axis="y", colors=muted_color, labelsize=9, length=0, pad=8)

ax.grid(
    color=border_color,
    linewidth=0.8,
    alpha=0.7,
    zorder=0
)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

legend = ax.legend(
    loc="lower right",
    frameon=False,
    fontsize=8
)
for label in legend.get_texts():
    label.set_color(text_color)

fig.text(
    0.5, 0.025,
    "Each point represents one fragrance. The gold line shows median rating within log-spaced vote-count ranges. "
    "Association does not imply causation.",
    color=muted_color,
    fontsize=7.5,
    ha="center"
)

plt.subplots_adjust(
    left=0.13,
    right=0.95,
    top=0.84,
    bottom=0.12
)

# ------------------------------------------
# Save PNG to Databricks Volume
# ------------------------------------------

if not os.path.isdir(OUTPUT_DIR):
    raise FileNotFoundError(
        f"Visualization directory does not exist: {OUTPUT_DIR}"
    )

fig.savefig(
    output_path,
    dpi=180,
    facecolor=fig.get_facecolor()
)

if not os.path.isfile(output_path):
    raise RuntimeError(f"Chart was not created: {output_path}")

print(f"Chart saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path):,} bytes")
print(f"PNG dimensions: {fig.canvas.get_width_height()}")
print(f"Fragrances plotted: {len(scatter_df):,}")
print(f"Spearman correlation: {rho:.4f}")
print(f"Spearman p-value: {p_value:.4g}")

plt.show()
plt.close(fig)


In [0]:

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# 1. Check duplicate and missing records
df = (
    rating_accords_df
    .select("id", "name", "rating_value", "rating_count")
    .toPandas()
)

print(f"Total rows: {len(df):,}")
print(f"Missing IDs: {df['id'].isna().sum():,}")
print(f"Duplicate IDs: {df['id'].duplicated().sum():,}")
print(f"Missing ratings: {df['rating_value'].isna().sum():,}")
print(f"Missing vote counts: {df['rating_count'].isna().sum():,}")

# Use one record per ID and retain valid numeric values
df["rating_value"] = pd.to_numeric(df["rating_value"], errors="coerce")
df["rating_count"] = pd.to_numeric(df["rating_count"], errors="coerce")

df = df.dropna(subset=["id", "rating_value", "rating_count"])
df = df.drop_duplicates(subset=["id"], keep="first")

# Vote counts must be positive for the logarithmic relationship
df = df[df["rating_count"] > 0].copy()

x = df["rating_count"].to_numpy(dtype=float)
y = df["rating_value"].to_numpy(dtype=float)

# 2. Calculate the observed Spearman correlation
observed_rho, p_value = spearmanr(x, y)

print(f"\nFragrances analyzed: {len(df):,}")
print(f"Spearman correlation: {observed_rho:.4f}")
print(f"P-value: {p_value:.3e}")

# 3. Bootstrap 95% confidence interval
# Resample fragrances with replacement.
# Use 1,000 iterations for a practical first estimate.
rng = np.random.default_rng(42)
n = len(df)
n_bootstrap = 1000
bootstrap_rhos = []

for i in range(n_bootstrap):
    indices = rng.integers(0, n, size=n)
    rho, _ = spearmanr(x[indices], y[indices])

    if np.isfinite(rho):
        bootstrap_rhos.append(rho)

ci_lower, ci_upper = np.percentile(
    bootstrap_rhos, [2.5, 97.5]
)

print(f"\n95% bootstrap CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Successful bootstrap samples: {len(bootstrap_rhos):,}")

if ci_upper < 0:
    print("Interpretation: the estimated association is negative.")
elif ci_lower > 0:
    print("Interpretation: the estimated association is positive.")
else:
    print("Interpretation: the interval includes zero.")


In [0]:

from pyspark.sql import functions as F

# Aggregate fragrance-level data by dominant accord
accord_stats = (
    rating_accords_df
    .filter(
        F.col("id").isNotNull()
        & F.col("dominant_accord").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
    )
    .groupBy("dominant_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.sum("rating_count").alias("total_votes"),
        F.avg("rating_value").alias("mean_rating"),
        F.expr(
            "percentile_approx(rating_value, 0.5)"
        ).alias("median_rating")
    )
    .filter(F.col("fragrance_count") >= 100)
)

# Select the 20 most-voted dominant accords
comparison = (
    accord_stats
    .orderBy(F.desc("total_votes"))
    .limit(20)
)

display(comparison)


In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# --------------------------------------------------
# 1. Prepare data — all 38 eligible dominant accords
# --------------------------------------------------
pdf = (
    accord_stats
    .filter("fragrance_count >= 100")
    .toPandas()
    .sort_values("total_votes", ascending=False)
    .reset_index(drop=True)
)

# --------------------------------------------------
# 2. Fragrance Intelligence palette
# --------------------------------------------------
BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "text.color": TEXT,
    "axes.labelcolor": MUTED,
    "xtick.color": MUTED,
    "ytick.color": TEXT,
})

# --------------------------------------------------
# 3. Create chart
# --------------------------------------------------
fig, ax1 = plt.subplots(figsize=(12, 13), dpi=180)
fig.patch.set_facecolor(BG)
ax1.set_facecolor(BG)

y = np.arange(len(pdf))

bars = ax1.barh(
    y,
    pdf["total_votes"],
    color=BROWN,
    height=0.60,
    alpha=0.95,
    zorder=2
)

ax1.set_yticks(y)
ax1.set_yticklabels(
    [
        f"{accord}  (n={count:,})"
        for accord, count in zip(
            pdf["dominant_accord"],
            pdf["fragrance_count"]
        )
    ],
    fontsize=9
)

ax1.invert_yaxis()

max_votes = pdf["total_votes"].max()
ax1.set_xlim(0, max_votes * 1.18)

# Total-vote labels inside bars, close to the y-axis
for bar, votes in zip(bars, pdf["total_votes"]):
    ax1.text(
        max_votes * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{votes:,.0f}",
        ha="left",
        va="center",
        color=TEXT,
        fontsize=8,
        fontweight="bold",
        zorder=4
    )

ax1.set_xlabel(
    "Total Community Votes by Dominant Accord",
    fontsize=10,
    labelpad=18,
    color=MUTED
)

ax1.xaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f"{value:,.0f}")
)

ax1.grid(
    axis="x",
    color=GRID,
    linewidth=0.8,
    alpha=0.8,
    zorder=0
)
ax1.set_axisbelow(True)

ax1.tick_params(axis="y", length=0, pad=8)
ax1.tick_params(axis="x", labelsize=8, pad=7)

for spine in ax1.spines.values():
    spine.set_color(GRID)

# --------------------------------------------------
# 4. Mean rating by dominant accord
# --------------------------------------------------
ax2 = ax1.twiny()
ax2.set_facecolor("none")
ax2.set_zorder(ax1.get_zorder() + 1)
ax2.patch.set_visible(False)

rating_min = pdf["mean_rating"].min()
rating_max = pdf["mean_rating"].max()
rating_range = rating_max - rating_min
padding = max(rating_range * 0.12, 0.02)

ax2.set_xlim(
    rating_min - padding,
    rating_max + padding
)

ax2.plot(
    pdf["mean_rating"],
    y,
    color=ROSE,
    linewidth=2.8,
    marker="o",
    markersize=7,
    markerfacecolor=GOLD,
    markeredgecolor=TEXT,
    markeredgewidth=1.1,
    solid_capstyle="round",
    zorder=10,
    clip_on=False
)

# Label each accord's mean rating
for rating, position in zip(pdf["mean_rating"], y):
    ax2.annotate(
        f"{rating:.2f}",
        xy=(rating, position),
        xytext=(8, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        color=TEXT,
        fontsize=7.5,
        fontweight="medium",
        bbox={
            "boxstyle": "round,pad=0.20",
            "facecolor": BG,
            "edgecolor": GRID,
            "linewidth": 0.6,
            "alpha": 0.98
        },
        zorder=12,
        clip_on=False
    )

ax2.set_xlabel(
    "Mean Community Rating by Dominant Accord",
    fontsize=10,
    labelpad=17,
    color=ROSE
)

ax2.tick_params(
    axis="x",
    colors=GOLD,
    labelsize=8,
    pad=7,
    length=3
)

ax2.xaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f"{value:.2f}")
)

for spine in ax2.spines.values():
    spine.set_color(GRID)

# --------------------------------------------------
# 5. Header and centered footer
# --------------------------------------------------
fig.text(
    0.08, 0.965,
    "FRAGRANCE INTELLIGENCE  /  RATINGS ANALYSIS",
    color=GOLD,
    fontsize=9,
    fontweight="bold"
)

fig.text(
    0.08, 0.925,
    "Total Votes and Mean Rating by Dominant Accord",
    color=TEXT,
    fontsize=19
)

fig.text(
    0.08, 0.892,
    "Bars show total community votes for each accord group; "
    "the line and labels show that group's mean fragrance rating.",
    color=MUTED,
    fontsize=9.5
)

fig.text(
    0.50, 0.035,
    f"All {len(pdf)} dominant accords with at least 100 fragrances are included.",
    color=MUTED,
    fontsize=8,
    ha="center",
    va="center"
)

plt.subplots_adjust(
    left=0.25,
    right=0.88,
    top=0.79,
    bottom=0.10
)

# --------------------------------------------------
# 6. Save
# --------------------------------------------------
output_path = (
    "/Volumes/main/fragrance_analytics/reference_data/"
    "visualizations/accord-total-votes-vs-mean-rating.png"
)

fig.savefig(
    output_path,
    dpi=180,
    facecolor=BG,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(f"Saved visualization: {output_path}")
print(f"Dominant accords included: {len(pdf)}")

In [0]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# --------------------------------------------------
# 1. Prepare data
# --------------------------------------------------
pdf = (
    accord_stats
    .filter("fragrance_count >= 100")
    .toPandas()
    .sort_values("total_votes", ascending=False)
    .reset_index(drop=True)
)

# --------------------------------------------------
# 2. Fragrance Intelligence palette
# --------------------------------------------------
BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "text.color": TEXT,
    "axes.labelcolor": MUTED,
    "xtick.color": MUTED,
    "ytick.color": TEXT,
})

# --------------------------------------------------
# 3. Create bubble chart
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 9), dpi=180)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

# Bubble area scales with fragrance count
counts = pdf["fragrance_count"].to_numpy()
min_size, max_size = 100, 1400

if counts.max() == counts.min():
    sizes = np.full(len(counts), (min_size + max_size) / 2)
else:
    sizes = min_size + (
        (counts - counts.min()) / (counts.max() - counts.min())
    ) * (max_size - min_size)

scatter = ax.scatter(
    pdf["total_votes"],
    pdf["mean_rating"],
    s=sizes,
    color=BROWN,
    alpha=0.78,
    edgecolors=GOLD,
    linewidths=1.4,
    zorder=3
)

# Accord labels
for row in pdf.itertuples(index=False):
    ax.annotate(
        row.dominant_accord,
        (row.total_votes, row.mean_rating),
        xytext=(6, 5),
        textcoords="offset points",
        fontsize=8,
        color=TEXT,
        bbox={
            "boxstyle": "round,pad=0.18",
            "facecolor": BG,
            "edgecolor": GRID,
            "linewidth": 0.5,
            "alpha": 0.92
        },
        zorder=4
    )

# Axes
ax.set_xlabel(
    "Total Community Votes by Dominant Accord",
    fontsize=10,
    labelpad=12,
    color=MUTED
)
ax.set_ylabel(
    "Mean Community Rating by Dominant Accord",
    fontsize=10,
    labelpad=12,
    color=MUTED
)

ax.xaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f"{value:,.0f}")
)

ax.grid(
    color=GRID,
    linewidth=0.8,
    alpha=0.8,
    zorder=0
)
ax.set_axisbelow(True)

ax.tick_params(axis="both", labelsize=8, pad=7)

for spine in ax.spines.values():
    spine.set_color(GRID)

# --------------------------------------------------
# 4. Bubble-size legend
# --------------------------------------------------
legend_counts = np.unique(
    np.quantile(counts, [0.25, 0.5, 0.75]).astype(int)
)

legend_handles = []
for count in legend_counts:
    if counts.max() == counts.min():
        size = (min_size + max_size) / 2
    else:
        size = min_size + (
            (count - counts.min()) / (counts.max() - counts.min())
        ) * (max_size - min_size)

    legend_handles.append(
        ax.scatter(
            [], [],
            s=size,
            color=BROWN,
            alpha=0.78,
            edgecolors=GOLD,
            linewidths=1.2,
            label=f"{count:,} fragrances"
        )
    )

legend = ax.legend(
    handles=legend_handles,
    title="Bubble size",
    loc="lower right",
    frameon=True,
    facecolor=BG,
    edgecolor=GRID,
    fontsize=8,
    title_fontsize=8.5,
    labelcolor=TEXT
)
legend.get_frame().set_alpha(0.98)
legend.get_title().set_color(MUTED)

# --------------------------------------------------
# 5. Header and footer
# --------------------------------------------------
fig.text(
    0.08, 0.965,
    "FRAGRANCE INTELLIGENCE  /  RATINGS ANALYSIS",
    color=GOLD,
    fontsize=9,
    fontweight="bold"
)

fig.text(
    0.08, 0.925,
    "Dominant Accord Popularity and Mean Rating",
    color=TEXT,
    fontsize=19
)

fig.text(
    0.08, 0.892,
    "Each bubble represents an accord group; larger bubbles contain more fragrances.",
    color=MUTED,
    fontsize=9.5
)

fig.text(
    0.50, 0.035,
    "Only accords with at least 100 fragrances are included. "
    "Votes are summed across each accord group.",
    color=MUTED,
    fontsize=8,
    ha="center",
    va="center"
)

plt.subplots_adjust(
    left=0.12,
    right=0.96,
    top=0.82,
    bottom=0.14
)

# --------------------------------------------------
# 6. Save
# --------------------------------------------------
output_path = (
    "/Volumes/main/fragrance_analytics/reference_data/"
    "visualizations/accord-total-votes-rating-bubble.png"
)

fig.savefig(
    output_path,
    dpi=180,
    facecolor=BG,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(f"Saved visualization: {output_path}")
print(f"Dominant accords included: {len(pdf)}")

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from matplotlib.ticker import FuncFormatter

# --------------------------------------------------
# 1. Aggregate data by dominant accord
# --------------------------------------------------
accord_popularity_df = (
    rating_accords_df
    .filter(
        F.col("id").isNotNull()
        & F.col("dominant_accord").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .groupBy("dominant_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.avg("rating_count").alias("mean_votes"),
        F.expr(
            "percentile_approx(rating_count, 0.5)"
        ).alias("median_votes"),
        F.avg("rating_value").alias("mean_rating")
    )
    .filter(F.col("fragrance_count") >= 100)
)

# --------------------------------------------------
# 2. Prepare data — all eligible dominant accords
# --------------------------------------------------
pdf = (
    accord_popularity_df
    .toPandas()
    .sort_values("mean_votes", ascending=False)
    .reset_index(drop=True)
)

# --------------------------------------------------
# 3. Fragrance Intelligence palette
# --------------------------------------------------
BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "text.color": TEXT,
    "axes.labelcolor": MUTED,
    "xtick.color": MUTED,
    "ytick.color": TEXT,
})

# --------------------------------------------------
# 4. Create chart
# --------------------------------------------------
fig, ax1 = plt.subplots(figsize=(12, 13), dpi=180)
fig.patch.set_facecolor(BG)
ax1.set_facecolor(BG)

y = np.arange(len(pdf))

bars = ax1.barh(
    y,
    pdf["mean_votes"],
    color=BROWN,
    height=0.60,
    alpha=0.95,
    zorder=2
)

ax1.set_yticks(y)
ax1.set_yticklabels(
    [
        f"{accord}  (n={count:,})"
        for accord, count in zip(
            pdf["dominant_accord"],
            pdf["fragrance_count"]
        )
    ],
    fontsize=9
)

ax1.invert_yaxis()

max_votes = pdf["mean_votes"].max()
ax1.set_xlim(0, max_votes * 1.18)

# Mean vote labels inside bars, near the y-axis
for bar, votes in zip(bars, pdf["mean_votes"]):
    ax1.text(
        max_votes * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{votes:,.0f}",
        ha="left",
        va="center",
        color=TEXT,
        fontsize=8,
        fontweight="bold",
        zorder=4
    )

ax1.set_xlabel(
    "Mean Community Votes per Fragrance",
    fontsize=10,
    labelpad=18,
    color=MUTED
)

ax1.xaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f"{value:,.0f}")
)

ax1.grid(
    axis="x",
    color=GRID,
    linewidth=0.8,
    alpha=0.8,
    zorder=0
)
ax1.set_axisbelow(True)

ax1.tick_params(axis="y", length=0, pad=8)
ax1.tick_params(axis="x", labelsize=8, pad=7)

for spine in ax1.spines.values():
    spine.set_color(GRID)

# --------------------------------------------------
# 5. Mean rating by dominant accord
# --------------------------------------------------
ax2 = ax1.twiny()
ax2.set_facecolor("none")
ax2.set_zorder(ax1.get_zorder() + 1)
ax2.patch.set_visible(False)

rating_min = pdf["mean_rating"].min()
rating_max = pdf["mean_rating"].max()
rating_range = rating_max - rating_min
padding = max(rating_range * 0.12, 0.02)

ax2.set_xlim(
    rating_min - padding,
    rating_max + padding
)

ax2.plot(
    pdf["mean_rating"],
    y,
    color=ROSE,
    linewidth=2.8,
    marker="o",
    markersize=7,
    markerfacecolor=GOLD,
    markeredgecolor=TEXT,
    markeredgewidth=1.1,
    solid_capstyle="round",
    zorder=10,
    clip_on=False
)

# Label each accord's mean rating
for rating, position in zip(pdf["mean_rating"], y):
    ax2.annotate(
        f"{rating:.2f}",
        xy=(rating, position),
        xytext=(8, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        color=TEXT,
        fontsize=7.5,
        fontweight="medium",
        bbox={
            "boxstyle": "round,pad=0.20",
            "facecolor": BG,
            "edgecolor": GRID,
            "linewidth": 0.6,
            "alpha": 0.98
        },
        zorder=12,
        clip_on=False
    )

ax2.set_xlabel(
    "Mean Community Rating by Dominant Accord",
    fontsize=10,
    labelpad=17,
    color=ROSE
)

ax2.tick_params(
    axis="x",
    colors=GOLD,
    labelsize=8,
    pad=7,
    length=3
)

ax2.xaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f"{value:.2f}")
)

for spine in ax2.spines.values():
    spine.set_color(GRID)

# --------------------------------------------------
# 6. Header and centered footer
# --------------------------------------------------
fig.text(
    0.08, 0.965,
    "FRAGRANCE INTELLIGENCE  /  RATINGS ANALYSIS",
    color=GOLD,
    fontsize=9,
    fontweight="bold"
)

fig.text(
    0.08, 0.925,
    "Mean Fragrance Popularity and Mean Rating by Dominant Accord",
    color=TEXT,
    fontsize=19
)

fig.text(
    0.08, 0.892,
    "Bars show mean community votes per fragrance; "
    "the line and labels show mean rating within each accord group.",
    color=MUTED,
    fontsize=9.5
)

fig.text(
    0.50, 0.035,
    f"All {len(pdf)} dominant accords with at least 100 fragrances are included. "
    "Votes and ratings use separate scales.",
    color=MUTED,
    fontsize=8,
    ha="center",
    va="center"
)

plt.subplots_adjust(
    left=0.25,
    right=0.88,
    top=0.79,
    bottom=0.10
)

# --------------------------------------------------
# 7. Save
# --------------------------------------------------
output_path = (
    "/Volumes/main/fragrance_analytics/reference_data/"
    "visualizations/accord-popularity-vs-mean-rating.png"
)

fig.savefig(
    output_path,
    dpi=180,
    facecolor=BG,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(f"Saved visualization: {output_path}")
print(f"Dominant accords included: {len(pdf)}")

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from matplotlib.ticker import FuncFormatter

# --------------------------------------------------
# 1. Aggregate data by dominant accord
# --------------------------------------------------
accord_popularity_df = (
    rating_accords_df
    .filter(
        F.col("id").isNotNull()
        & F.col("dominant_accord").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .groupBy("dominant_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.avg("rating_count").alias("mean_votes"),
        F.avg("rating_value").alias("mean_rating")
    )
    .filter(F.col("fragrance_count") >= 100)
)

# --------------------------------------------------
# 2. Prepare data for visualization
# --------------------------------------------------
pdf = (
    accord_popularity_df
    .toPandas()
    .sort_values("mean_votes", ascending=False)
    .reset_index(drop=True)
)

# --------------------------------------------------
# 3. Fragrance Intelligence palette
# --------------------------------------------------
BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "text.color": TEXT,
    "axes.labelcolor": MUTED,
    "xtick.color": MUTED,
    "ytick.color": TEXT,
})

# --------------------------------------------------
# 4. Create bubble chart
# --------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 9), dpi=180)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

# Bubble area represents the number of fragrances
counts = pdf["fragrance_count"].to_numpy()
min_size, max_size = 100, 1400

if counts.max() == counts.min():
    sizes = np.full(len(counts), (min_size + max_size) / 2)
else:
    sizes = min_size + (
        (counts - counts.min()) / (counts.max() - counts.min())
    ) * (max_size - min_size)

ax.scatter(
    pdf["mean_votes"],
    pdf["mean_rating"],
    s=sizes,
    color=BROWN,
    alpha=0.78,
    edgecolors=GOLD,
    linewidths=1.4,
    zorder=3
)

# Label each dominant accord
for row in pdf.itertuples(index=False):
    ax.annotate(
        row.dominant_accord,
        (row.mean_votes, row.mean_rating),
        xytext=(6, 5),
        textcoords="offset points",
        fontsize=8,
        color=TEXT,
        bbox={
            "boxstyle": "round,pad=0.18",
            "facecolor": BG,
            "edgecolor": GRID,
            "linewidth": 0.5,
            "alpha": 0.92
        },
        zorder=4
    )

# --------------------------------------------------
# 5. Axes and formatting
# --------------------------------------------------
ax.set_xlabel(
    "Mean Community Votes per Fragrance",
    fontsize=10,
    labelpad=12,
    color=MUTED
)

ax.set_ylabel(
    "Mean Community Rating",
    fontsize=10,
    labelpad=12,
    color=MUTED
)

ax.xaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f"{value:,.0f}")
)

ax.grid(
    color=GRID,
    linewidth=0.8,
    alpha=0.8,
    zorder=0
)
ax.set_axisbelow(True)
ax.tick_params(axis="both", labelsize=8, pad=7)

for spine in ax.spines.values():
    spine.set_color(GRID)

# --------------------------------------------------
# 6. Bubble-size legend
# --------------------------------------------------
legend_counts = np.unique(
    np.quantile(counts, [0.25, 0.5, 0.75]).astype(int)
)

legend_handles = []

for count in legend_counts:
    if counts.max() == counts.min():
        size = (min_size + max_size) / 2
    else:
        size = min_size + (
            (count - counts.min()) / (counts.max() - counts.min())
        ) * (max_size - min_size)

    legend_handles.append(
        ax.scatter(
            [],
            [],
            s=size,
            color=BROWN,
            alpha=0.78,
            edgecolors=GOLD,
            linewidths=1.2,
            label=f"{count:,} fragrances"
        )
    )

legend = ax.legend(
    handles=legend_handles,
    title="Bubble size",
    loc="lower right",
    frameon=True,
    facecolor=BG,
    edgecolor=GRID,
    fontsize=8,
    title_fontsize=8.5,
    labelcolor=TEXT
)

legend.get_frame().set_alpha(0.98)
legend.get_title().set_color(MUTED)

# --------------------------------------------------
# 7. Header and footer
# --------------------------------------------------
fig.text(
    0.08, 0.965,
    "FRAGRANCE INTELLIGENCE  /  RATINGS ANALYSIS",
    color=GOLD,
    fontsize=9,
    fontweight="bold"
)

fig.text(
    0.08, 0.925,
    "Mean Fragrance Popularity and Rating by Dominant Accord",
    color=TEXT,
    fontsize=19
)

fig.text(
    0.08, 0.892,
    "Each bubble represents a dominant accord; larger bubbles contain more fragrances.",
    color=MUTED,
    fontsize=9.5
)

fig.text(
    0.50, 0.035,
    f"All {len(pdf)} dominant accords with at least 100 fragrances are included. "
    "Popularity and rating are means across individual fragrances.",
    color=MUTED,
    fontsize=8,
    ha="center",
    va="center"
)

plt.subplots_adjust(
    left=0.12,
    right=0.96,
    top=0.82,
    bottom=0.14
)

# --------------------------------------------------
# 8. Save
# --------------------------------------------------
output_path = (
    "/Volumes/main/fragrance_analytics/reference_data/"
    "visualizations/accord-mean-votes-rating-bubble.png"
)

fig.savefig(
    output_path,
    dpi=180,
    facecolor=BG,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(f"Saved visualization: {output_path}")
print(f"Dominant accords included: {len(pdf)}")

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Aggregate fragrance-level metrics by dominant accord
# --------------------------------------------------
accord_interpretation_df = (
    rating_accords_df
    .filter(
        F.col("id").isNotNull()
        & F.col("dominant_accord").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .groupBy("dominant_accord")
    .agg(
        # Number of distinct fragrances
        F.countDistinct("id").alias("fragrance_count"),

        # Popularity: each fragrance contributes equally
        F.avg("rating_count").alias("mean_votes"),
        F.expr(
            "percentile_approx(rating_count, 0.5, 10000)"
        ).alias("median_votes"),
        F.min("rating_count").alias("min_votes"),
        F.max("rating_count").alias("max_votes"),
        F.sum("rating_count").alias("total_votes"),

        # Rating: each fragrance contributes equally
        F.avg("rating_value").alias("mean_rating"),
        F.expr(
            "percentile_approx(rating_value, 0.5, 10000)"
        ).alias("median_rating"),
        F.min("rating_value").alias("min_rating"),
        F.max("rating_value").alias("max_rating")
    )
    .filter(F.col("fragrance_count") >= 100)
)

# --------------------------------------------------
# 2. Add overall reference values
# --------------------------------------------------
overall_stats = (
    rating_accords_df
    .filter(
        F.col("id").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .agg(
        F.countDistinct("id").alias("all_fragrances"),
        F.avg("rating_value").alias("overall_mean_rating"),
        F.avg("rating_count").alias("overall_mean_votes"),
        F.expr(
            "percentile_approx(rating_value, 0.5, 10000)"
        ).alias("overall_median_rating"),
        F.expr(
            "percentile_approx(rating_count, 0.5, 10000)"
        ).alias("overall_median_votes")
    )
)

# --------------------------------------------------
# 3. Convert to pandas and calculate comparisons
# --------------------------------------------------
pdf = accord_interpretation_df.toPandas()
overall = overall_stats.first().asDict()

pdf["rating_vs_overall"] = (
    pdf["mean_rating"] - overall["overall_mean_rating"]
)

pdf["votes_vs_overall_pct"] = (
    (pdf["mean_votes"] / overall["overall_mean_votes"]) - 1
) * 100

pdf["mean_median_votes_ratio"] = (
    pdf["mean_votes"] / pdf["median_votes"].replace(0, float("nan"))
)

pdf["mean_median_rating_gap"] = (
    pdf["mean_rating"] - pdf["median_rating"]
)

# --------------------------------------------------
# 4. Sort and display all eligible accords
# --------------------------------------------------
pdf = (
    pdf.sort_values("mean_votes", ascending=False)
    .reset_index(drop=True)
)

display(
    spark.createDataFrame(pdf)
    .select(
        "dominant_accord",
        "fragrance_count",
        "total_votes",
        "mean_votes",
        "median_votes",
        "min_votes",
        "max_votes",
        "mean_median_votes_ratio",
        "mean_rating",
        "median_rating",
        "min_rating",
        "max_rating",
        "rating_vs_overall",
        "votes_vs_overall_pct",
        "mean_median_rating_gap"
    )
)

# --------------------------------------------------
# 5. Print overall reference metrics
# --------------------------------------------------
print(f"All rated fragrances: {overall['all_fragrances']:,}")
print(f"Overall mean rating: {overall['overall_mean_rating']:.3f}")
print(f"Overall median rating: {overall['overall_median_rating']:.3f}")
print(f"Overall mean votes per fragrance: {overall['overall_mean_votes']:.2f}")
print(f"Overall median votes per fragrance: {overall['overall_median_votes']:.0f}")
print(f"Dominant accords included: {len(pdf)}")

# --------------------------------------------------
# 6. Export interpretation dataset to CSV
# --------------------------------------------------
output_path = (
    "/Volumes/main/fragrance_analytics/reference_data/"
    "visualizations/accord-interpretation-data.csv"
)

pdf.to_csv(output_path, index=False)

print(f"Interpretation data saved to: {output_path}")

## Accord-Pair Analysis and Segmentation

This analysis groups fragrances by their ordered dominant and secondary accords and calculates pair-level metrics, including fragrance count, total and average votes, mean and median ratings, and the share of fragrances rated 4.5 or higher. Each fragrance contributes its own rating and vote count to the aggregation for its corresponding accord pair.

Pairs with at least 30 fragrances are retained for the initial analysis. A separate visualization applies a higher minimum threshold of 100 fragrances to improve interpretability and reduce the influence of smaller groups.

The **Popularity–Rating Matrix** segments eligible accord pairs using the median mean-vote count and median mean-rating across pairs as reference lines. This produces four relative segments:

- **Popular Performers:** Above-median popularity and above-median rating.
- **High-Rated Niche Pairs:** Below-median popularity and above-median rating.
- **Popular Underperformers:** Above-median popularity and below-median rating.
- **Low-Visibility, Lower-Rated Pairs:** Below-median popularity and below-median rating.

These labels describe relative positions within the analyzed dataset, not absolute market performance. Popularity is estimated using community votes rather than sales, and ratings are aggregated from individual fragrance-level ratings. The segmentation is exploratory and helps identify accord combinations for further investigation.

One methodological distinction worth preserving: the metrics are calculated from fragrance-level observations and then aggregated into accord pairs. The visualization's bubbles represent aggregated pairs, not individual fragrances.

In [0]:
from pyspark.sql import functions as F

# Create metrics for dominant + second accord pairs
accord_pair_stats = (
    rating_accords_df
    .filter(
        F.col("id").isNotNull()
        & F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .groupBy("dominant_accord", "second_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.sum("rating_count").alias("total_votes"),
        F.avg("rating_count").alias("mean_votes"),
        F.expr(
            "percentile_approx(rating_count, 0.5)"
        ).alias("median_votes"),
        F.avg("rating_value").alias("mean_rating"),
        F.expr(
            "percentile_approx(rating_value, 0.5)"
        ).alias("median_rating"),
        F.avg(
            F.when(F.col("rating_value") >= 4.5, 1.0)
             .otherwise(0.0)
        ).alias("share_rated_4_5_plus")
    )
    .filter(F.col("fragrance_count") >= 30)
    .withColumn(
        "accord_pair",
        F.concat_ws(" + ", "dominant_accord", "second_accord")
    )
    .orderBy(
        F.desc("fragrance_count"),
        F.desc("mean_rating")
    )
)

display(accord_pair_stats)

In [0]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap

# --------------------------------------------------
# FRAGRANCE INTELLIGENCE / ACCORD PAIR SEGMENTATION
# Unit of analysis: ordered dominant + second accord pair
# --------------------------------------------------

BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"
FONT = "DejaVu Sans"

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    OUTPUT_DIR, "accord-pair-segmentation.png"
)

MIN_FRAGRANCES = 100
LABELS_PER_QUADRANT = 6
LABELS_PER_BAND = 2
HIGH_RATING_THRESHOLD = 4.10
LOW_RATING_THRESHOLD = 3.85

# --------------------------------------------------
# PREPARE DATA
# --------------------------------------------------

plot_df = (
    accord_pair_stats
    .filter(f"fragrance_count >= {MIN_FRAGRANCES}")
    .toPandas()
)

plot_df = plot_df.dropna(
    subset=[
        "dominant_accord",
        "second_accord",
        "mean_votes",
        "mean_rating",
        "fragrance_count"
    ]
).copy()

# Preserve the order: dominant accord + second accord
plot_df["accord_pair"] = (
    plot_df["dominant_accord"].str.title()
    + " + "
    + plot_df["second_accord"].str.title()
)

plot_df = plot_df.drop_duplicates(
    subset=["dominant_accord", "second_accord"]
).reset_index(drop=True)

if len(plot_df) < 4:
    raise ValueError("Not enough eligible accord pairs to segment.")

# --------------------------------------------------
# SEGMENT PAIRS USING MEDIAN REFERENCE LINES
# Use >= consistently for both the chart and summary.
# --------------------------------------------------

x_median = plot_df["mean_votes"].median()
y_median = plot_df["mean_rating"].median()

plot_df["popularity_segment"] = np.where(
    plot_df["mean_votes"] >= x_median,
    "Higher popularity",
    "Lower popularity"
)

plot_df["rating_segment"] = np.where(
    plot_df["mean_rating"] >= y_median,
    "Higher rating",
    "Lower rating"
)

segment_names = {
    ("Higher popularity", "Higher rating"):
        "Popular Performers",

    ("Lower popularity", "Higher rating"):
        "High-Rated Niche Pairs",

    ("Higher popularity", "Lower rating"):
        "Popular Underperformers",

    ("Lower popularity", "Lower rating"):
        "Low-Visibility, Lower-Rated Pairs"
}

plot_df["segment"] = [
    segment_names[(popularity, rating)]
    for popularity, rating in zip(
        plot_df["popularity_segment"],
        plot_df["rating_segment"]
    )
]

# Use segment names consistently for labeling and summaries.
plot_df["quadrant"] = plot_df["segment"]

segment_order = [
    "Popular Performers",
    "High-Rated Niche Pairs",
    "Popular Underperformers",
    "Low-Visibility, Lower-Rated Pairs"
]

# --------------------------------------------------
# BUBBLE SIZES
# Bubble area represents the number of fragrances in a pair.
# --------------------------------------------------

counts = plot_df["fragrance_count"].to_numpy(dtype=float)
size_min, size_max = 65, 750

if counts.max() == counts.min():
    sizes = np.full(
        len(counts),
        (size_min + size_max) / 2
    )
else:
    sizes = size_min + (
        (counts - counts.min())
        / (counts.max() - counts.min())
    ) * (size_max - size_min)

plot_df["bubble_size"] = sizes

# --------------------------------------------------
# FIGURE AND HEADER
# --------------------------------------------------

fig, ax = plt.subplots(
    figsize=(16, 12),
    dpi=180,
    facecolor=BG
)
ax.set_facecolor(BG)

fig.suptitle(
    "Accord Pair Segmentation",
    x=0.10,
    y=0.985,
    ha="left",
    color=TEXT,
    fontsize=23,
    fontfamily=FONT,
    fontweight="bold"
)

fig.text(
    0.10,
    0.948,
    (
        f"{len(plot_df):,} ordered accord pairs  |  "
        f"Minimum {MIN_FRAGRANCES} fragrances per pair  |  "
        "Bubble size = number of fragrances"
    ),
    ha="left",
    color=MUTED,
    fontsize=10,
    fontfamily=FONT
)

# --------------------------------------------------
# SCATTERPLOT
# --------------------------------------------------

ax.grid(
    color=GRID,
    linewidth=0.7,
    alpha=0.65
)
ax.set_axisbelow(True)

cmap = LinearSegmentedColormap.from_list(
    "plum_gold",
    [BROWN, "#80606b", GOLD]
)

ax.scatter(
    plot_df["mean_votes"],
    plot_df["mean_rating"],
    s=plot_df["bubble_size"],
    c=plot_df["mean_rating"],
    cmap=cmap,
    alpha=0.90,
    edgecolors=BG,
    linewidths=0.9,
    zorder=3
)

# Median reference lines
ax.axvline(
    x_median,
    color=GOLD,
    linestyle=(0, (5, 4)),
    linewidth=1.6,
    alpha=0.95,
    zorder=2
)

ax.axhline(
    y_median,
    color=GOLD,
    linestyle=(0, (5, 4)),
    linewidth=1.6,
    alpha=0.95,
    zorder=2
)

# Axis limits with room around the bubbles
x_min = plot_df["mean_votes"].min()
x_max = plot_df["mean_votes"].max()
y_min = plot_df["mean_rating"].min()
y_max = plot_df["mean_rating"].max()

x_range = max(x_max - x_min, 1)
y_range = max(y_max - y_min, 0.05)

ax.set_xlim(
    max(0, x_min - 0.05 * x_range),
    x_max + 0.12 * x_range
)
ax.set_ylim(
    y_min - 0.12 * y_range,
    y_max + 0.12 * y_range
)

# --------------------------------------------------
# SEGMENT LABELS
# --------------------------------------------------

quadrant_style = dict(
    color=MUTED,
    fontsize=7.5,
    fontfamily=FONT,
    alpha=0.95,
    linespacing=1.5,
    zorder=4
)

# Position the segment names in their respective quadrants.
segment_positions = {
    "Popular Performers": (0.98, 0.97, "right", "top"),
    "High-Rated Niche Pairs": (0.02, 0.97, "left", "top"),
    "Popular Underperformers": (0.98, 0.03, "right", "bottom"),
    "Low-Visibility, Lower-Rated Pairs": (
        0.02, 0.03, "left", "bottom"
    )
}

for segment, (x, y, ha, va) in segment_positions.items():
    ax.text(
        x,
        y,
        segment.upper(),
        transform=ax.transAxes,
        ha=ha,
        va=va,
        **quadrant_style
    )

# --------------------------------------------------
# SELECT LABELS ACROSS THE VERTICAL RANGE
# Select up to LABELS_PER_QUADRANT base labels per segment.
# High- and low-rating threshold pairs are added afterward.
# --------------------------------------------------

label_candidates = plot_df.copy()
label_candidates["vertical_band"] = np.nan

for segment, group in label_candidates.groupby("segment"):
    if len(group) == 1:
        label_candidates.loc[group.index, "vertical_band"] = 1
    else:
        ranks = group["mean_rating"].rank(method="first")
        band_count = min(3, len(group))

        bands = pd.qcut(
            ranks,
            q=band_count,
            labels=False,
            duplicates="drop"
        )

        label_candidates.loc[group.index, "vertical_band"] = (
            bands.to_numpy()
        )

selected_groups = []

# Choose the largest pairs in each vertical band.
for segment, group in label_candidates.groupby("segment"):
    for _, band_group in group.groupby("vertical_band"):
        selected_groups.append(
            band_group
            .sort_values("fragrance_count", ascending=False)
            .head(LABELS_PER_BAND)
        )

if selected_groups:
    base_labels = (
        pd.concat(selected_groups)
        .drop_duplicates(subset=["accord_pair"])
    )
else:
    base_labels = label_candidates.iloc[0:0].copy()

# Enforce the base-label limit within each segment.
base_labels = (
    base_labels
    .sort_values("fragrance_count", ascending=False)
    .groupby("segment", group_keys=False)
    .head(LABELS_PER_QUADRANT)
)

# Add more labels only when a segment has fewer than the target.
additional_labels = []

for segment, group in label_candidates.groupby("segment"):
    already_selected = base_labels[
        base_labels["segment"] == segment
    ]

    remaining_slots = (
        LABELS_PER_QUADRANT - len(already_selected)
    )

    if remaining_slots > 0:
        remaining = group[
            ~group["accord_pair"].isin(base_labels["accord_pair"])
        ]

        additional_labels.append(
            remaining
            .sort_values("fragrance_count", ascending=False)
            .head(remaining_slots)
        )

if additional_labels:
    base_labels = pd.concat(
        [base_labels] + additional_labels
    ).drop_duplicates(subset=["accord_pair"])

# --------------------------------------------------
# ADD HIGH- AND LOW-RATING PAIRS
# These are additional labels, independent of base-label limits.
# --------------------------------------------------

threshold_mask = (
    (plot_df["mean_rating"] > HIGH_RATING_THRESHOLD)
    | (plot_df["mean_rating"] < LOW_RATING_THRESHOLD)
)

threshold_labels = plot_df[threshold_mask].copy()

label_df = (
    pd.concat([base_labels, threshold_labels])
    .drop_duplicates(subset=["accord_pair"])
)

# --------------------------------------------------
# ANNOTATE BUBBLES
# Gold labels highlight pairs beyond the rating thresholds.
# --------------------------------------------------

for segment, group in label_df.groupby("segment", sort=False):
    group = group.sort_values("mean_rating")

    for j, (_, row) in enumerate(group.iterrows()):

        # Point labels toward the inside of the chart.
        if row["popularity_segment"] == "Higher popularity":
            dx, ha = -10, "right"
        else:
            dx, ha = 10, "left"

        # Alternate labels above and below the bubbles.
        if j % 2 == 0:
            dy, va = 13, "bottom"
        else:
            dy, va = -16, "top"

        is_threshold_pair = (
            row["mean_rating"] > HIGH_RATING_THRESHOLD
            or row["mean_rating"] < LOW_RATING_THRESHOLD
        )

        label_color = GOLD if is_threshold_pair else TEXT
        border_color = GOLD if is_threshold_pair else GRID
        arrow_color = GOLD if is_threshold_pair else MUTED

        ax.annotate(
            row["accord_pair"],
            xy=(row["mean_votes"], row["mean_rating"]),
            xytext=(dx, dy),
            textcoords="offset points",
            ha=ha,
            va=va,
            color=label_color,
            fontsize=7.2,
            fontweight="bold" if is_threshold_pair else "normal",
            fontfamily=FONT,
            bbox=dict(
                boxstyle="round,pad=0.28",
                facecolor=BG,
                edgecolor=border_color,
                alpha=0.97,
                linewidth=0.8 if is_threshold_pair else 0.6
            ),
            arrowprops=dict(
                arrowstyle="-",
                color=arrow_color,
                linewidth=0.75 if is_threshold_pair else 0.6,
                alpha=0.85
            ),
            annotation_clip=False,
            zorder=5
        )

# --------------------------------------------------
# AXES
# --------------------------------------------------

ax.set_xlabel(
    "Mean votes per fragrance within pair",
    color=MUTED,
    fontsize=11,
    labelpad=12,
    fontfamily=FONT
)

ax.set_ylabel(
    "Mean fragrance rating within pair",
    color=MUTED,
    fontsize=11,
    labelpad=12,
    fontfamily=FONT
)

ax.tick_params(
    colors=MUTED,
    labelsize=9
)

for spine in ax.spines.values():
    spine.set_color(GRID)

# --------------------------------------------------
# BUBBLE SIZE LEGEND
# --------------------------------------------------

legend_counts = sorted({
    int(counts.min()),
    int(np.median(counts)),
    int(counts.max())
})

legend_handles = []

for count in legend_counts:
    if counts.max() == counts.min():
        marker_area = (size_min + size_max) / 2
    else:
        marker_area = (
            size_min
            + (count - counts.min())
            / (counts.max() - counts.min())
            * (size_max - size_min)
        )

    legend_handles.append(
        Line2D(
            [0], [0],
            marker="o",
            linestyle="none",
            markerfacecolor=GOLD,
            markeredgecolor=BG,
            markersize=np.sqrt(marker_area),
            label=f"{count:,}"
        )
    )

ax.legend(
    handles=legend_handles,
    title="Fragrances per pair",
    loc="center left",
    bbox_to_anchor=(1.015, 0.5),
    frameon=False,
    labelcolor=MUTED,
    title_fontsize=9,
    fontsize=8
)

# --------------------------------------------------
# EXPLANATION OF REFERENCE LINES
# --------------------------------------------------

fig.text(
    0.50,
    0.050,
    "HOW TO READ THE DASHED LINES",
    ha="center",
    color=GOLD,
    fontsize=9,
    fontfamily=FONT,
    fontweight="bold"
)

fig.text(
    0.50,
    0.026,
    (
        f"Vertical = median mean votes per fragrance within pair "
        f"({x_median:,.1f})"
        f"     |     Horizontal = median mean fragrance rating "
        f"({y_median:.3f})"
    ),
    ha="center",
    color=MUTED,
    fontsize=8.5,
    fontfamily=FONT
)

fig.subplots_adjust(
    left=0.10,
    right=0.81,
    top=0.90,
    bottom=0.14
)

# --------------------------------------------------
# SAVE AND DISPLAY
# --------------------------------------------------

fig.savefig(
    OUTPUT_PATH,
    dpi=180,
    facecolor=BG,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# --------------------------------------------------
# SUMMARY
# --------------------------------------------------

print(f"Saved chart: {OUTPUT_PATH}")
print(f"Eligible ordered accord pairs: {len(plot_df):,}")
print(f"Total labeled pairs: {len(label_df):,}")

print(
    f"Pairs with mean rating > {HIGH_RATING_THRESHOLD:.2f}: "
    f"{(plot_df['mean_rating'] > HIGH_RATING_THRESHOLD).sum():,}"
)

print(
    f"Pairs with mean rating < {LOW_RATING_THRESHOLD:.2f}: "
    f"{(plot_df['mean_rating'] < LOW_RATING_THRESHOLD).sum():,}"
)

print(f"Median mean votes per fragrance within pair: {x_median:,.1f}")
print(f"Median mean fragrance rating within pair: {y_median:.3f}")

segment_summary = (
    plot_df
    .groupby("segment", as_index=False)
    .agg(
        pair_count=("accord_pair", "count"),
        median_fragrance_count=("fragrance_count", "median"),
        average_rating=("mean_rating", "mean"),
        average_votes_per_fragrance=("mean_votes", "mean")
    )
)

segment_summary["segment_order"] = (
    segment_summary["segment"].map(
        {name: i for i, name in enumerate(segment_order)}
    )
)

segment_summary = (
    segment_summary
    .sort_values("segment_order")
    .drop(columns="segment_order")
    .reset_index(drop=True)
)

display(segment_summary)

In [0]:

# Assign every eligible accord pair to one of the four segments
segmentation_df = plot_df.copy()

segmentation_df["popularity_segment"] = np.where(
    segmentation_df["mean_votes"] > x_median,
    "Popular",
    "Less Popular"
)

segmentation_df["rating_segment"] = np.where(
    segmentation_df["mean_rating"] > y_median,
    "Higher-Rated",
    "Lower-Rated"
)

# Name the four segments using the Popularity–Rating Matrix
segment_names = {
    ("Popular", "Higher-Rated"): "Popular Performers",
    ("Less Popular", "Higher-Rated"): "High-Rated Niche Pairs",
    ("Popular", "Lower-Rated"): "Popular Underperformers",
    ("Less Popular", "Lower-Rated"): "Low-Visibility, Lower-Rated Pairs"
}

segmentation_df["segment"] = segmentation_df.apply(
    lambda row: segment_names[
        (row["popularity_segment"], row["rating_segment"])
    ],
    axis=1
)

# Keep the most useful columns and sort within each segment
segmentation_df = segmentation_df[
    [
        "segment",
        "accord_pair",
        "fragrance_count",
        "mean_votes",
        "mean_rating",
        "median_rating",
        "total_votes",
        "share_rated_4_5_plus"
    ]
].sort_values(
    ["segment", "mean_rating", "mean_votes"],
    ascending=[True, False, False]
).reset_index(drop=True)

# Display all pairs, with the segment shown first
display(segmentation_df)

## Statistical Analysis: Accord-Pair Popularity and Rating

### Objective

This analysis examines the relationship between community engagement and average fragrance ratings at the **ordered accord-pair level**, rather than at the individual-fragrance level.

Each observation represents a dominant + secondary accord pair. The metrics used are the mean number of community votes per fragrance (`mean_votes`) and the mean fragrance rating within each pair (`mean_rating`).

### Results

The Spearman rank correlation was calculated across 147 eligible accord pairs:

- **Spearman correlation:** 0.2135
- **P-value:** 0.00942
- **Direction and strength:** Weak positive association

The results indicate that accord pairs with higher average votes per fragrance tend to have somewhat higher average fragrance ratings. The association is statistically significant at the 1% level under the standard test assumptions, but its strength is limited.

### Interpretation and Limitations

This result differs from the weak negative correlation previously observed at the individual-fragrance level. The findings are not necessarily contradictory because the analyses use different units of observation: individual fragrances in the first analysis and aggregated accord pairs in this analysis.

The pair-level result should be interpreted as exploratory. Accord pairs may share fragrances or accord structures, so the independence assumption of the standard correlation test may not fully hold. Statistical significance does not establish causation or imply a strong practical relationship.

The analysis describes an association between average community engagement and average ratings across accord pairs; it does not demonstrate that popularity causes higher ratings.

In [0]:

from scipy.stats import spearmanr
import pandas as pd

# Use the same minimum pair size as the existing segmentation
pair_corr_df = (
    accord_pair_stats
    .filter(F.col("fragrance_count") >= 100)
    .select(
        "accord_pair",
        "fragrance_count",
        "mean_votes",
        "mean_rating"
    )
    .dropna(subset=["mean_votes", "mean_rating"])
    .toPandas()
)

result = spearmanr(
    pair_corr_df["mean_votes"],
    pair_corr_df["mean_rating"]
)

print(f"Eligible accord pairs: {len(pair_corr_df):,}")
print(f"Spearman correlation: {result.statistic:.4f}")
print(f"P-value: {result.pvalue:.4g}")

In [0]:
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

# ============================================================
# Configuration
# ============================================================

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Project color palette
BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"
PLUM = "#80606b"

SEGMENT_COLORS = {
    "Popular Performers": GOLD,
    "High-Rated Niche Pairs": ROSE,
    "Popular Underperformers": BROWN,
    "Low-Visibility, Lower-Rated Pairs": PLUM,
}

SEGMENT_FILES = {
    "Popular Performers": "popular-performers.png",
    "High-Rated Niche Pairs": "high-rated-niche-pairs.png",
    "Popular Underperformers": "popular-underperformers.png",
    "Low-Visibility, Lower-Rated Pairs":
        "low-visibility-lower-rated-pairs.png",
}

SEGMENT_DEFINITIONS = {
    "Popular Performers":
        "Above-median popularity and above-median rating",
    "High-Rated Niche Pairs":
        "Below-median popularity and above-median rating",
    "Popular Underperformers":
        "Above-median popularity and below-median rating",
    "Low-Visibility, Lower-Rated Pairs":
        "Below-median popularity and below-median rating",
}

# Typography
PAIR_LABEL_SIZE = 14
VALUE_LABEL_SIZE = 14
X_TICK_SIZE = 12
TITLE_SIZE = 16

# ============================================================
# Recalculate segment classification
# ============================================================

# Use the same eligible pair population as the segmentation matrix
x_median = plot_df["mean_votes"].median()
y_median = plot_df["mean_rating"].median()

segmentation_df = plot_df.copy()

# Values equal to the median are classified in the higher segment
segmentation_df["popularity_segment"] = np.where(
    segmentation_df["mean_votes"] >= x_median,
    "Popular",
    "Less Popular",
)

segmentation_df["rating_segment"] = np.where(
    segmentation_df["mean_rating"] >= y_median,
    "Higher-Rated",
    "Lower-Rated",
)

segment_names = {
    ("Popular", "Higher-Rated"): "Popular Performers",
    ("Less Popular", "Higher-Rated"): "High-Rated Niche Pairs",
    ("Popular", "Lower-Rated"): "Popular Underperformers",
    ("Less Popular", "Lower-Rated"):
        "Low-Visibility, Lower-Rated Pairs",
}

segmentation_df["segment"] = [
    segment_names[(popularity, rating)]
    for popularity, rating in zip(
        segmentation_df["popularity_segment"],
        segmentation_df["rating_segment"],
    )
]

# Validate required columns
required_cols = {
    "segment",
    "accord_pair",
    "fragrance_count",
    "share_rated_4_5_plus",
}

missing_cols = required_cols - set(segmentation_df.columns)

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# ============================================================
# Generate four separate visualizations
# ============================================================

for segment, filename in SEGMENT_FILES.items():

    segment_df = segmentation_df[
        segmentation_df["segment"] == segment
    ].copy()

    if segment_df.empty:
        print(f"Skipping empty segment: {segment}")
        continue

    # Rank by share rated 4.5+, then by sample size
    segment_df = segment_df.sort_values(
        ["share_rated_4_5_plus", "fragrance_count"],
        ascending=[False, False],
    ).head(20)

    # Reverse for horizontal bars: highest value at the top
    segment_df = segment_df.iloc[::-1]

    # Increase height to give larger labels sufficient breathing room
    fig_height = max(8, len(segment_df) * 0.48 + 2)

    fig, ax = plt.subplots(figsize=(15, fig_height))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)

    bars = ax.barh(
        segment_df["accord_pair"],
        segment_df["share_rated_4_5_plus"],
        color=SEGMENT_COLORS[segment],
        edgecolor=BG,
        height=0.72,
    )

    # Percentage and fragrance-count labels
    max_share = segment_df["share_rated_4_5_plus"].max()
    label_offset = max(max_share * 0.015, 0.003)

    for bar, (_, row) in zip(bars, segment_df.iterrows()):
        value = row["share_rated_4_5_plus"]

        ax.text(
            value + label_offset,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.1%}  (n={int(row['fragrance_count'])})",
            va="center",
            ha="left",
            color=TEXT,
            fontsize=VALUE_LABEL_SIZE,
        )

    # Title and segment definition
    ax.set_title(
        f"{segment}\n{SEGMENT_DEFINITIONS[segment]}",
        color=TEXT,
        fontsize=TITLE_SIZE,
        fontweight="bold",
        pad=20,
        linespacing=1.7,
    )

    # Axis labels
    ax.set_xlabel(
        "Share of fragrances rated 4.5 or higher",
        color=MUTED,
        fontsize=12,
        labelpad=14,
    )

    ax.set_ylabel(
        "Dominant + second accord",
        color=MUTED,
        fontsize=12,
        labelpad=12,
    )

    # Larger accord-pair names and readable axis ticks
    ax.tick_params(
        axis="y",
        colors=TEXT,
        labelsize=PAIR_LABEL_SIZE,
        length=0,
        pad=8,
    )

    ax.tick_params(
        axis="x",
        colors=MUTED,
        labelsize=X_TICK_SIZE,
    )

    # Extra room to the right for percentage/count labels
    ax.set_xlim(
        0,
        max_share * 1.45 if max_share > 0 else 1,
    )

    ax.xaxis.set_major_formatter(
        mtick.PercentFormatter(xmax=1.0)
    )

    # Subtle grid and clean borders
    ax.grid(
        axis="x",
        color=GRID,
        linestyle="--",
        linewidth=0.7,
        alpha=0.8,
    )
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()

    output_path = os.path.join(OUTPUT_DIR, filename)

    plt.savefig(
        output_path,
        dpi=250,
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )

    plt.show()
    plt.close(fig)

    print(f"Saved: {output_path}")

# ============================================================
# Report classification thresholds
# ============================================================

print("\nSegment classification thresholds")
print("---------------------------------")
print(f"Popularity median (mean votes): {x_median:.2f}")
print(f"Rating median (mean rating):    {y_median:.3f}")
print("\nAll four segment visualizations have been generated.")

In [0]:
from pyspark.sql import functions as F

# ============================================================
# Accord triad statistics
# ============================================================

accord_triad_stats = (
    rating_accords_df
    .filter(
        F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
        & F.col("third_accord").isNotNull()
        & F.col("rating_value").isNotNull()
    )
    .groupBy(
        "dominant_accord",
        "second_accord",
        "third_accord",
    )
    .agg(
        F.countDistinct("id").alias("fragrance_count"),

        F.sum("rating_count").alias("total_votes"),

        F.avg("rating_count").alias("mean_votes"),

        F.expr("percentile_approx(rating_count, 0.5)")
         .alias("median_votes"),

        F.avg("rating_value").alias("mean_rating"),

        F.expr("percentile_approx(rating_value, 0.5)")
         .alias("median_rating"),

        F.avg(
            F.when(F.col("rating_value") >= 4.5, 1.0)
             .otherwise(0.0)
        ).alias("share_rated_4_5_plus"),
    )
    .withColumn(
        "accord_triad",
        F.concat_ws(
            " + ",
            F.initcap("dominant_accord"),
            F.initcap("second_accord"),
            F.initcap("third_accord"),
        ),
    )
)

display(
    accord_triad_stats
    .orderBy(F.desc("fragrance_count"))
    .limit(20)
)

In [0]:
triad_summary = accord_triad_stats.agg(
    F.count("*").alias("unique_triads"),
    F.sum("fragrance_count").alias("total_triad_assignments"),
    F.min("fragrance_count").alias("min_fragrances_per_triad"),
    F.max("fragrance_count").alias("max_fragrances_per_triad"),
)

display(triad_summary)

In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. Validate fragrance-level uniqueness
# ============================================================

duplicate_ids = (
    rating_accords_df
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
)

print("Fragrance-level validation")
print("---------------------------")
print(
    "Distinct fragrance IDs:",
    rating_accords_df.select("id").distinct().count()
)
print(
    "Rows in source:",
    rating_accords_df.count()
)
print(
    "IDs appearing more than once:",
    duplicate_ids.count()
)

display(duplicate_ids.orderBy(F.desc("count")).limit(20))


# ============================================================
# 2. Preliminary rankings
# ============================================================

MIN_FRAGRANCES = 50

eligible_triads = accord_triad_stats.filter(
    F.col("fragrance_count") >= MIN_FRAGRANCES
)

print(f"\nTriads with at least {MIN_FRAGRANCES} fragrances:")
print(eligible_triads.count())


# A. Most frequent triads
print("\nA. Most frequent triads")
display(
    eligible_triads
    .orderBy(
        F.desc("fragrance_count"),
        F.desc("mean_rating")
    )
    .select(
        "accord_triad",
        "fragrance_count",
        "mean_votes",
        "mean_rating",
        "share_rated_4_5_plus"
    )
    .limit(20)
)


# B. Most popular triads by average votes per fragrance
print("\nB. Most popular triads")
display(
    eligible_triads
    .orderBy(
        F.desc("mean_votes"),
        F.desc("fragrance_count")
    )
    .select(
        "accord_triad",
        "fragrance_count",
        "mean_votes",
        "median_votes",
        "mean_rating",
        "share_rated_4_5_plus"
    )
    .limit(20)
)


# C. Highest-rated triads
print("\nC. Highest-rated triads")
display(
    eligible_triads
    .orderBy(
        F.desc("mean_rating"),
        F.desc("fragrance_count")
    )
    .select(
        "accord_triad",
        "fragrance_count",
        "mean_rating",
        "median_rating",
        "mean_votes",
        "share_rated_4_5_plus"
    )
    .limit(20)
)


# D. Highest share rated 4.5+
print("\nD. Triads with the highest share rated 4.5+")
display(
    eligible_triads
    .orderBy(
        F.desc("share_rated_4_5_plus"),
        F.desc("fragrance_count")
    )
    .select(
        "accord_triad",
        "fragrance_count",
        "share_rated_4_5_plus",
        "mean_rating",
        "mean_votes"
    )
    .limit(20)
)

In [0]:

import os
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# --------------------------------------------------
# Configuration
# --------------------------------------------------

OUTPUT_DIR = "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"
PLUM = "#80606b"

plt.rcParams["font.family"] = "DejaVu Sans"

# Convert the eligible Spark DataFrame to Pandas once
triads_pdf = eligible_triads.toPandas()

# --------------------------------------------------
# Reusable plotting function
# --------------------------------------------------

def plot_triad_ranking(
    data,
    metric,
    title,
    filename,
    bar_color,
    value_format="number",
    ascending=True
):
    plot_df = (
        data
        .sort_values(metric, ascending=ascending)
        .tail(20)
        .copy()
    )

    # Keep the requested ranking order in the displayed chart
    plot_df = plot_df.sort_values(metric, ascending=True)

    if plot_df.empty:
        print(f"No data available for: {title}")
        return

    labels = plot_df["accord_triad"].astype(str).tolist()
    values = plot_df[metric].astype(float).tolist()
    counts = plot_df["fragrance_count"].astype(int).tolist()

    fig_height = max(8, len(plot_df) * 0.48 + 2)

    fig, ax = plt.subplots(
        figsize=(15, fig_height),
        facecolor=BG
    )
    ax.set_facecolor(BG)

    bars = ax.barh(
        labels,
        values,
        color=bar_color,
        edgecolor=BG,
        linewidth=0.7,
        height=0.72
    )

    # Center title across the entire figure
    fig.suptitle(
        title,
        x=0.5,
        y=0.985,
        ha="center",
        va="top",
        color=TEXT,
        fontsize=17,
        fontweight="bold"
    )

    # Axis styling
    ax.set_xlabel(
        {
            "fragrance_count": "Number of fragrances",
            "mean_votes": "Mean votes per fragrance",
            "mean_rating": "Mean fragrance rating",
            "share_rated_4_5_plus": "Share rated 4.5+"
        }.get(metric, metric),
        color=MUTED,
        fontsize=13,
        labelpad=12
    )

    ax.tick_params(
        axis="y",
        colors=TEXT,
        labelsize=14,
        length=0,
        pad=9
    )

    ax.tick_params(
        axis="x",
        colors=MUTED,
        labelsize=12,
        length=0,
        pad=7
    )

    ax.xaxis.grid(
        True,
        color=GRID,
        linewidth=0.8,
        alpha=0.7
    )
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(False)

    # Format values and leave space for labels
    max_value = max(values) if values else 1
    ax.set_xlim(0, max_value * 1.42)

    if value_format == "percentage":
        ax.xaxis.set_major_formatter(
            FuncFormatter(lambda x, _: f"{x:.0%}")
        )
        label_offset = max_value * 0.012
    elif value_format == "rating":
        ax.xaxis.set_major_formatter(
            FuncFormatter(lambda x, _: f"{x:.1f}")
        )
        label_offset = max_value * 0.012
    elif value_format == "votes":
        ax.xaxis.set_major_formatter(
            FuncFormatter(lambda x, _: f"{x:,.0f}")
        )
        label_offset = max_value * 0.012
    else:
        ax.xaxis.set_major_formatter(
            FuncFormatter(lambda x, _: f"{x:,.0f}")
        )
        label_offset = max_value * 0.012

    # Value labels and fragrance counts
    for bar, value, count in zip(bars, values, counts):
        if value_format == "percentage":
            value_label = f"{value:.1%}"
        elif value_format == "rating":
            value_label = f"{value:.3f}"
        elif value_format == "votes":
            value_label = f"{value:,.1f}"
        else:
            value_label = f"{value:,.0f}"

        ax.text(
            value + label_offset,
            bar.get_y() + bar.get_height() / 2,
            f"{value_label}  (n={count:,})",
            va="center",
            ha="left",
            color=TEXT,
            fontsize=14
        )

    # Reserve space for the centered title and axis labels
    fig.subplots_adjust(
        left=0.36,
        right=0.94,
        top=0.91,
        bottom=0.09
    )

    output_path = os.path.join(OUTPUT_DIR, filename)

    fig.savefig(
        output_path,
        dpi=250,
        facecolor=BG,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(f"Saved: {output_path}")


# --------------------------------------------------
# A. Most frequent triads
# --------------------------------------------------

plot_triad_ranking(
    data=triads_pdf,
    metric="fragrance_count",
    title="Most Frequent Fragrance Accord Triads",
    filename="accord-triads-most-frequent.png",
    bar_color=GOLD,
    value_format="number",
    ascending=True
)


# --------------------------------------------------
# B. Most popular triads by mean votes
# --------------------------------------------------

plot_triad_ranking(
    data=triads_pdf,
    metric="mean_votes",
    title="Most Popular Fragrance Accord Triads",
    filename="accord-triads-most-popular.png",
    bar_color=ROSE,
    value_format="votes",
    ascending=True
)


# --------------------------------------------------
# C. Highest-rated triads
# --------------------------------------------------

plot_triad_ranking(
    data=triads_pdf,
    metric="mean_rating",
    title="Highest-Rated Fragrance Accord Triads",
    filename="accord-triads-highest-rated.png",
    bar_color=BROWN,
    value_format="rating",
    ascending=True
)


# --------------------------------------------------
# D. Highest share of fragrances rated 4.5+
# --------------------------------------------------

plot_triad_ranking(
    data=triads_pdf,
    metric="share_rated_4_5_plus",
    title="Triads with the Highest Share of Fragrances Rated 4.5+",
    filename="accord-triads-highest-rated-share.png",
    bar_color=PLUM,
    value_format="percentage",
    ascending=True
)

# Fragrance Accord Segmentation — Methodology Notes

## Purpose

Document the segmentation method used to analyze fragrance accord pairs and ordered triads, so the same approach can be reproduced and interpreted consistently.

## Data sources and eligibility

- **Fragrance-level source:** `rating_accords_df`
- **Triad-level aggregated data:** `accord_triad_stats`
- **Current eligible triads:** `eligible_triads`
- **Minimum support threshold:** 50 fragrances per ordered triad.
- Each row in the fragrance-level dataset represents a unique fragrance. The source was already deduplicated and validated: 62,623 rows, 62,623 unique IDs, and zero duplicate IDs.

Triads are **ordered combinations** of dominant, second, and third accords. For example, `Amber + Woody + Warm Spicy` and `Woody + Amber + Warm Spicy` are treated as different triads because the accord positions differ.

The triad analysis includes only fragrances with non-null dominant, second, and third accords and a non-null rating.

## Metrics

For each eligible pair or triad, the aggregated metrics include:

- `fragrance_count`: number of fragrances assigned to the pair or triad.
- `mean_votes`: average `rating_count` per fragrance. This is used as a **popularity proxy**, not as sales or market share.
- `mean_rating`: average fragrance rating.
- `share_rated_4_5_plus`: proportion of fragrances in the pair or triad with `rating_value >= 4.5`.
- Additional descriptive metrics may include total votes and median votes/rating.

The chart label `n=` means the number of fragrances in the pair or triad, not the number of votes.

## Segmentation method

Use two dimensions:

1. **Popularity:** `mean_votes`
2. **Rating:** `mean_rating`

Calculate the median of each metric across the eligible set being segmented. Classify each pair or triad as **High** when its value is greater than or equal to the relevant median, and **Low** when it is below the median.

| Segment | Popularity | Rating | Interpretation |
|---|---|---|---|
| Popular Performers | High | High | Above-or-at-median popularity and rating |
| High-Rated Niche Pairs / Triads | Low | High | Lower popularity, higher rating |
| Popular Underperformers | High | Low | Higher popularity, lower rating |
| Low-Visibility, Lower-Rated Pairs / Triads | Low | Low | Below-median popularity and rating |

For triads, the segment names are:

- `Popular Performers`
- `High-Rated Niche Triads`
- `Popular Underperformers`
- `Low-Visibility, Lower-Rated Triads`

For pairs, use the equivalent pair labels.

### Important implementation detail

Compute median thresholds on the eligible pairs or triads used in that specific analysis. Do not assume pair medians and triad medians are interchangeable. The segmentation is relative to the analyzed set, not an absolute quality standard.

## Segment visualizations

Create a separate horizontal bar chart for each segment.

- Rank the top 20 combinations by `share_rated_4_5_plus`, descending.
- Display the bars in ascending order so the highest value appears at the top.
- Label each bar with the percentage of fragrances rated 4.5+ and `(n=fragrance_count)`.
- Use the project's established palette:
  - Background: `#21171f`
  - Text: `#f4ece9`
  - Muted text: `#c6b6c0`
  - Gold: `#d6b47a`
  - Brown: `#98765b`
  - Rose: `#d4a1b2`
  - Grid: `#4d3948`
  - Plum: `#80606b`
- Use DejaVu Sans, readable 14 pt combination labels and value labels, centered figure titles, and high-resolution PNG exports at 250 dpi.
- Save charts under:
  `/Volumes/main/fragrance_analytics/reference_data/visualizations/`

## Interpretation cautions

1. `mean_votes` is a popularity proxy based on rating counts. It does **not** measure fragrance sales or market share.
2. Mean votes can be skewed by a few exceptionally popular fragrances. Consider median votes alongside mean votes when interpreting a combination.
3. `share_rated_4_5_plus` is the proportion of fragrances rated at least 4.5, not the share of all votes.
4. A high share rated 4.5+ and a high mean rating are related but distinct measures; show them as complementary metrics.
5. Segment summary averages that average each combination equally are **unweighted across combinations**. They should not be described as fragrance-weighted overall averages.
6. Results depend on the minimum-support filter and median thresholds. Record any changes to the threshold or eligible population when comparing analyses.
7. The current triad threshold is `fragrance_count >= 50`, which produced 159 eligible ordered triads out of 11,382 unique ordered triads in the initial triad aggregation.

## Useful observations from the initial triad analysis

- `Vanilla + Fruity + Sweet` had the highest mean votes in the top-popularity ranking (about 609 votes per fragrance), while its median votes were only 32. This suggests a skewed distribution influenced by a few very popular fragrances.
- `Amber + Warm Spicy + Woody` had the highest mean rating in the supplied top-rated results (about 4.227, with 130 fragrances) and a 23.8% share rated 4.5+.
- `Woody + Powdery + Citrus` had the highest share rated 4.5+ in the supplied results (about 25.4%, with 59 fragrances), despite a mean rating of about 3.986.
- Warm, woody, and amber combinations appeared repeatedly among the high-rating and high-rating-share triads. Treat this as a descriptive pattern, not evidence of causation.

## Output files for triad rankings

The initial four ranking charts are saved as:

- `accord-triads-most-frequent.png`
- `accord-triads-most-popular.png`
- `accord-triads-highest-rated.png`
- `accord-triads-highest-rated-share.png`

The four segment chart filenames are generated from each segment name in lowercase with spaces replaced by hyphens and commas removed, followed by `-triads.png`.

## Reproducibility checklist

- [ ] Confirm the source and aggregation DataFrames are the expected versions.
- [ ] Confirm the minimum-support threshold.
- [ ] Calculate medians from the eligible set.
- [ ] Assign each combination to exactly one segment.
- [ ] Check the count of combinations in each segment.
- [ ] Generate one chart per segment with consistent formatting.
- [ ] Keep the interpretation notes and threshold values with the project outputs.

In [0]:

import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# --------------------------------------------------
# Configuration
# --------------------------------------------------

OUTPUT_DIR = "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"
PLUM = "#80606b"

plt.rcParams["font.family"] = "DejaVu Sans"

# Use the existing eligible triads dataset
triads_pdf = eligible_triads.toPandas().copy()

# --------------------------------------------------
# 1. Calculate median thresholds
# --------------------------------------------------

popularity_median = triads_pdf["mean_votes"].median()
rating_median = triads_pdf["mean_rating"].median()

print(f"Popularity threshold (median mean votes): {popularity_median:.2f}")
print(f"Rating threshold (median mean rating):   {rating_median:.3f}")

# --------------------------------------------------
# 2. Assign each triad to a segment
# --------------------------------------------------

triads_pdf["popularity_level"] = triads_pdf["mean_votes"].apply(
    lambda x: "High" if x >= popularity_median else "Low"
)

triads_pdf["rating_level"] = triads_pdf["mean_rating"].apply(
    lambda x: "High" if x >= rating_median else "Low"
)

def assign_segment(row):
    if row["popularity_level"] == "High" and row["rating_level"] == "High":
        return "Popular Performers"

    elif row["popularity_level"] == "Low" and row["rating_level"] == "High":
        return "High-Rated Niche Triads"

    elif row["popularity_level"] == "High" and row["rating_level"] == "Low":
        return "Popular Underperformers"

    else:
        return "Low-Visibility, Lower-Rated Triads"

triads_pdf["segment"] = triads_pdf.apply(assign_segment, axis=1)

# --------------------------------------------------
# 3. Summarize segment membership
# --------------------------------------------------

segment_order = [
    "Popular Performers",
    "High-Rated Niche Triads",
    "Popular Underperformers",
    "Low-Visibility, Lower-Rated Triads"
]

segment_summary = (
    triads_pdf
    .groupby("segment", observed=False)
    .agg(
        triad_count=("accord_triad", "count"),
        fragrance_count=("fragrance_count", "sum"),
        mean_votes=("mean_votes", "mean"),
        mean_rating=("mean_rating", "mean"),
        mean_high_rating_share=("share_rated_4_5_plus", "mean")
    )
    .reindex(segment_order)
    .reset_index()
)

display(segment_summary)

print("\nTriads per segment:")
print(
    triads_pdf["segment"]
    .value_counts()
    .reindex(segment_order, fill_value=0)
)

# --------------------------------------------------
# 4. Visualize each segment separately
#    Ranked by share rated 4.5+
# --------------------------------------------------

segment_colors = {
    "Popular Performers": GOLD,
    "High-Rated Niche Triads": ROSE,
    "Popular Underperformers": BROWN,
    "Low-Visibility, Lower-Rated Triads": PLUM
}

def plot_segment(segment_name, color, top_n=20):

    plot_df = (
        triads_pdf[triads_pdf["segment"] == segment_name]
        .nlargest(top_n, "share_rated_4_5_plus")
        .sort_values("share_rated_4_5_plus", ascending=True)
        .copy()
    )

    if plot_df.empty:
        print(f"No triads found for: {segment_name}")
        return

    fig_height = max(7, len(plot_df) * 0.48 + 2)

    fig, ax = plt.subplots(
        figsize=(15, fig_height),
        facecolor=BG
    )
    ax.set_facecolor(BG)

    bars = ax.barh(
        plot_df["accord_triad"],
        plot_df["share_rated_4_5_plus"],
        color=color,
        edgecolor=BG,
        linewidth=0.7,
        height=0.72
    )

    # Center title across the entire figure
    fig.suptitle(
        f"{segment_name}\nTop Triads by Share Rated 4.5+",
        x=0.5,
        y=0.985,
        ha="center",
        va="top",
        color=TEXT,
        fontsize=17,
        fontweight="bold"
    )

    ax.set_xlabel(
        "Share of fragrances rated 4.5+",
        color=MUTED,
        fontsize=13,
        labelpad=12
    )

    ax.xaxis.set_major_formatter(PercentFormatter(1.0))

    ax.tick_params(
        axis="y",
        colors=TEXT,
        labelsize=14,
        length=0,
        pad=9
    )

    ax.tick_params(
        axis="x",
        colors=MUTED,
        labelsize=12,
        length=0,
        pad=7
    )

    ax.xaxis.grid(
        True,
        color=GRID,
        linewidth=0.8,
        alpha=0.7
    )
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(False)

    max_value = plot_df["share_rated_4_5_plus"].max()
    ax.set_xlim(0, max_value * 1.45 if max_value > 0 else 1)

    # Add percentage and fragrance-count labels
    for bar, (_, row) in zip(bars, plot_df.iterrows()):
        value = row["share_rated_4_5_plus"]
        count = int(row["fragrance_count"])

        ax.text(
            value + max_value * 0.012,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.1%}  (n={count:,})",
            va="center",
            ha="left",
            color=TEXT,
            fontsize=14
        )

    fig.subplots_adjust(
        left=0.36,
        right=0.94,
        top=0.89,
        bottom=0.09
    )

    filename = (
        segment_name.lower()
        .replace(",", "")
        .replace(" ", "-")
        + "-triads.png"
    )

    output_path = os.path.join(OUTPUT_DIR, filename)

    fig.savefig(
        output_path,
        dpi=250,
        facecolor=BG,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(f"Saved: {output_path}")


# --------------------------------------------------
# 5. Generate all four segment charts
# --------------------------------------------------

for segment_name in segment_order:
    plot_segment(
        segment_name=segment_name,
        color=segment_colors[segment_name],
        top_n=20
    )

In [0]:

import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# --------------------------------------------------
# Configuration
# --------------------------------------------------

OUTPUT_DIR = "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"
PLUM = "#80606b"

plt.rcParams["font.family"] = "DejaVu Sans"

N_LARGEST_TRIADS = 10
N_EXTREMES_PER_METRIC = 10
MAX_LABELS = 35
LABEL_FONT_SIZE = 11
LABEL_BACKGROUND_ALPHA = 0.55

# --------------------------------------------------
# 1. Prepare eligible triads
# --------------------------------------------------

triads_pdf = eligible_triads.toPandas().copy()

triads_pdf = triads_pdf.dropna(
    subset=["mean_votes", "mean_rating", "fragrance_count"]
).copy()

popularity_median = triads_pdf["mean_votes"].median()
rating_median = triads_pdf["mean_rating"].median()

print(f"Eligible triads: {len(triads_pdf):,}")
print(f"Median mean votes: {popularity_median:,.2f}")
print(f"Median mean rating: {rating_median:.3f}")

# --------------------------------------------------
# 2. Assign segmentation categories
# --------------------------------------------------

def assign_segment(row):
    high_popularity = row["mean_votes"] >= popularity_median
    high_rating = row["mean_rating"] >= rating_median

    if high_popularity and high_rating:
        return "Popular Performers"
    elif not high_popularity and high_rating:
        return "High-Rated Niche Triads"
    elif high_popularity and not high_rating:
        return "Popular Underperformers"
    else:
        return "Low-Visibility, Lower-Rated Triads"


triads_pdf["segment"] = triads_pdf.apply(
    assign_segment,
    axis=1
)

segment_colors = {
    "Popular Performers": GOLD,
    "High-Rated Niche Triads": ROSE,
    "Popular Underperformers": BROWN,
    "Low-Visibility, Lower-Rated Triads": PLUM
}

# --------------------------------------------------
# 3. Scale bubbles by fragrance count
# --------------------------------------------------

min_size = 55
max_size = 950

counts = triads_pdf["fragrance_count"].astype(float)

if counts.max() == counts.min():
    triads_pdf["bubble_size"] = min_size
else:
    triads_pdf["bubble_size"] = (
        min_size
        + (
            (counts - counts.min())
            / (counts.max() - counts.min())
        ) ** 0.5
        * (max_size - min_size)
    )

# --------------------------------------------------
# 4. Select high/low extremes and largest triads
# --------------------------------------------------

# Highest and lowest mean votes
highest_votes = triads_pdf.nlargest(
    N_EXTREMES_PER_METRIC, "mean_votes"
)
lowest_votes = triads_pdf.nsmallest(
    N_EXTREMES_PER_METRIC, "mean_votes"
)

# Highest and lowest mean ratings
highest_ratings = triads_pdf.nlargest(
    N_EXTREMES_PER_METRIC, "mean_rating"
)
lowest_ratings = triads_pdf.nsmallest(
    N_EXTREMES_PER_METRIC, "mean_rating"
)

# Largest triads by number of fragrances
largest_triads = triads_pdf.nlargest(
    N_LARGEST_TRIADS, "fragrance_count"
)

# Combine selections and remove duplicates
label_candidates = pd.concat([
    highest_votes,
    lowest_votes,
    highest_ratings,
    lowest_ratings,
    largest_triads
]).drop_duplicates(subset=["accord_triad"]).copy()

# Prioritize largest triads, then extremes in either metric
label_candidates["is_largest"] = label_candidates["accord_triad"].isin(
    largest_triads["accord_triad"]
)

label_candidates["vote_extremeness"] = (
    (label_candidates["mean_votes"] - popularity_median).abs()
    / max(abs(popularity_median), 1)
)

label_candidates["rating_extremeness"] = (
    (label_candidates["mean_rating"] - rating_median).abs()
)

label_candidates = label_candidates.sort_values(
    by=[
        "is_largest",
        "vote_extremeness",
        "rating_extremeness"
    ],
    ascending=[False, False, False]
)

label_candidates = label_candidates.head(MAX_LABELS)

print(f"Triads selected for labels: {len(label_candidates)}")

display(
    label_candidates[
        [
            "accord_triad",
            "fragrance_count",
            "mean_votes",
            "mean_rating",
            "segment"
        ]
    ].sort_values("fragrance_count", ascending=False)
)

# --------------------------------------------------
# 5. Create the bubble chart
# --------------------------------------------------

fig, ax = plt.subplots(
    figsize=(19, 13),
    facecolor=BG
)
ax.set_facecolor(BG)

for segment_name, color in segment_colors.items():
    subset = triads_pdf[
        triads_pdf["segment"] == segment_name
    ]

    if subset.empty:
        continue

    ax.scatter(
        subset["mean_votes"],
        subset["mean_rating"],
        s=subset["bubble_size"],
        color=color,
        alpha=0.76,
        edgecolors=TEXT,
        linewidths=0.5,
        label=segment_name,
        zorder=2
    )

# Median reference lines
ax.axvline(
    popularity_median,
    color=TEXT,
    linestyle="--",
    linewidth=1.4,
    alpha=0.8,
    zorder=1
)

ax.axhline(
    rating_median,
    color=TEXT,
    linestyle="--",
    linewidth=1.4,
    alpha=0.8,
    zorder=1
)

# --------------------------------------------------
# 6. Add larger, semi-transparent labels
# --------------------------------------------------

for _, row in label_candidates.iterrows():
    x = row["mean_votes"]
    y = row["mean_rating"]
    label = row["accord_triad"]
    n = int(row["fragrance_count"])

    label_text = f"{label}\n(n={n:,})"

    ax.annotate(
        label_text,
        xy=(x, y),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=LABEL_FONT_SIZE,
        color=TEXT,
        ha="left",
        va="bottom",
        bbox=dict(
            boxstyle="round,pad=0.32",
            facecolor=BG,
            edgecolor=GRID,
            alpha=LABEL_BACKGROUND_ALPHA
        ),
        arrowprops=dict(
            arrowstyle="-",
            color=MUTED,
            linewidth=0.8,
            alpha=0.8
        ),
        zorder=5
    )

# --------------------------------------------------
# 7. Titles and axis formatting
# --------------------------------------------------

fig.suptitle(
    "Fragrance Accord Triads: Popularity vs. Rating",
    x=0.5,
    y=0.985,
    ha="center",
    color=TEXT,
    fontsize=19,
    fontweight="bold"
)

ax.set_xlabel(
    "Mean votes per fragrance",
    color=MUTED,
    fontsize=14,
    labelpad=12
)

ax.set_ylabel(
    "Mean fragrance rating",
    color=MUTED,
    fontsize=14,
    labelpad=12
)

ax.tick_params(
    axis="both",
    colors=MUTED,
    labelsize=12,
    length=0,
    pad=8
)

ax.xaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f"{x:,.0f}")
)

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda y, _: f"{y:.2f}")
)

ax.xaxis.grid(
    True,
    color=GRID,
    linewidth=0.8,
    alpha=0.7
)

ax.yaxis.grid(
    True,
    color=GRID,
    linewidth=0.8,
    alpha=0.7
)

ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

# --------------------------------------------------
# 8. Segment legend
# --------------------------------------------------

legend = ax.legend(
    title="Triad segment",
    loc="upper left",
    frameon=True,
    facecolor=BG,
    edgecolor=GRID,
    fontsize=11,
    title_fontsize=12,
    labelcolor=TEXT
)

legend.get_title().set_color(TEXT)

# --------------------------------------------------
# 9. Bubble-size legend
# --------------------------------------------------

reference_counts = sorted(
    set([
        int(counts.min()),
        int(counts.median()),
        int(counts.max())
    ])
)

size_handles = []

for count in reference_counts:
    if counts.max() == counts.min():
        reference_size = min_size
    else:
        reference_size = (
            min_size
            + (
                (count - counts.min())
                / (counts.max() - counts.min())
            ) ** 0.5
            * (max_size - min_size)
        )

    handle = ax.scatter(
        [],
        [],
        s=reference_size,
        color=MUTED,
        alpha=0.65,
        edgecolors=TEXT,
        linewidths=0.5,
        label=f"{count:,} fragrances"
    )
    size_handles.append(handle)

size_legend = ax.legend(
    handles=size_handles,
    title="Bubble size",
    loc="lower right",
    frameon=True,
    facecolor=BG,
    edgecolor=GRID,
    fontsize=10,
    title_fontsize=11,
    labelcolor=TEXT
)

size_legend.get_title().set_color(TEXT)
ax.add_artist(legend)

# --------------------------------------------------
# 10. Save and display
# --------------------------------------------------

fig.subplots_adjust(
    left=0.10,
    right=0.97,
    top=0.92,
    bottom=0.11
)

output_path = os.path.join(
    OUTPUT_DIR,
    "accord-triads-popularity-rating-segmentation-labelled.png"
)

fig.savefig(
    output_path,
    dpi=250,
    facecolor=BG,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(f"Saved: {output_path}")


In [0]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = (
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

TOP_N = 20

# Consistent visual style
BG = "#21171f"
TEXT = "#f4ece9"
MUTED = "#c6b6c0"
GOLD = "#d6b47a"
BROWN = "#98765b"
ROSE = "#d4a1b2"
GRID = "#4d3948"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "text.color": TEXT,
    "axes.labelcolor": TEXT,
    "xtick.color": MUTED,
    "ytick.color": TEXT,
    "figure.facecolor": BG,
    "axes.facecolor": BG,
})

# ============================================================
# PREPARE DATA
# ============================================================

triads_pdf = eligible_triads.toPandas().copy()

# Ensure the metrics used in the charts are numeric
metrics = [
    "mean_votes",
    "mean_rating",
    "fragrance_count",
]

for metric in metrics:
    triads_pdf[metric] = pd.to_numeric(
        triads_pdf[metric],
        errors="coerce"
    )

# Keep rows with valid values for the metrics being analyzed
triads_pdf = triads_pdf.dropna(
    subset=["dominant_accord", "second_accord", "third_accord"]
).copy()

print(f"Eligible triads available for analysis: {len(triads_pdf):,}")
print(f"Number of triads displayed per chart: {TOP_N}")

# ============================================================
# HELPER FUNCTION
# ============================================================

def plot_triad_metric(
    data,
    metric,
    title,
    xlabel,
    filename,
    bar_color,
    value_format="number",
    ranking="highest",
):
    """
    Create a horizontal bar chart for the highest or lowest triads.

    ranking='highest':
        Select the TOP_N highest values and display them in descending
        order, with the highest value at the top.

    ranking='lowest':
        Select the TOP_N lowest values and display them in ascending
        order, with the lowest value at the top.
    """

    if ranking not in ("highest", "lowest"):
        raise ValueError(
            "ranking must be either 'highest' or 'lowest'"
        )

    # Use only records with valid values for the selected metric
    metric_data = data.dropna(subset=[metric]).copy()

    # Select the top or bottom N triads
    if ranking == "highest":
        selected = metric_data.nlargest(TOP_N, metric)
        sort_ascending = False
    else:
        selected = metric_data.nsmallest(TOP_N, metric)
        sort_ascending = True

    # Highest charts: descending order.
    # Lowest charts: ascending order.
    # The first item will be displayed at the top.
    plot_df = (
        selected
        .sort_values(
            metric,
            ascending=sort_ascending
        )
        .reset_index(drop=True)
    )

    if plot_df.empty:
        print(f"Skipping {filename}: no valid data.")
        return

    # Build readable labels
    plot_df["triad_label"] = (
        plot_df["dominant_accord"].astype(str)
        + " + "
        + plot_df["second_accord"].astype(str)
        + " + "
        + plot_df["third_accord"].astype(str)
    )

    # Add fragrance counts to labels
    plot_df["display_label"] = (
        plot_df["triad_label"]
        + "  (n="
        + plot_df["fragrance_count"].round(0).astype(int).astype(str)
        + ")"
    )

    # Reverse vertical positions so the first sorted row appears at the top
    y_positions = np.arange(len(plot_df))[::-1]

    fig_height = max(8, len(plot_df) * 0.48 + 2)

    fig, ax = plt.subplots(
        figsize=(16, fig_height)
    )

    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)

    bars = ax.barh(
        y_positions,
        plot_df[metric],
        color=bar_color,
        edgecolor=BG,
        linewidth=0.7,
        height=0.72,
    )

    # Axis labels
    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        plot_df["display_label"],
        fontsize=12,
    )

    ax.set_xlabel(
        xlabel,
        fontsize=13,
        labelpad=12,
    )

    # Centered chart title
    fig.suptitle(
        title,
        fontsize=20,
        fontweight="bold",
        color=TEXT,
        x=0.5,
        y=0.98,
        ha="center",
    )

    # Grid and spines
    ax.xaxis.grid(
        True,
        linestyle="--",
        linewidth=0.7,
        color=GRID,
        alpha=0.8,
    )

    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.tick_params(
        axis="y",
        length=0,
        pad=10,
    )

    ax.tick_params(
        axis="x",
        labelsize=11,
    )

    # Add value labels at the end of each bar
    max_value = plot_df[metric].max()
    min_value = plot_df[metric].min()
    value_range = max_value - min_value

    # Keep enough space for labels, including when values are similar
    if value_range == 0:
        label_offset = max(abs(max_value) * 0.015, 0.01)
    else:
        label_offset = value_range * 0.015

    for bar, value in zip(bars, plot_df[metric]):
        if value_format == "rating":
            label = f"{value:.3f}"
        elif value_format == "integer":
            label = f"{value:,.0f}"
        else:
            label = f"{value:,.2f}"

        ax.text(
            value + label_offset,
            bar.get_y() + bar.get_height() / 2,
            label,
            va="center",
            ha="left",
            fontsize=11,
            color=TEXT,
        )

    # Leave room for value labels on the right
    data_min = min(0, float(plot_df[metric].min()))
    data_max = float(plot_df[metric].max())

    if data_max == data_min:
        data_max = data_min + 1

    padding = max((data_max - data_min) * 0.16, 0.05)

    ax.set_xlim(
        left=data_min,
        right=data_max + padding,
    )

    plt.tight_layout(
        rect=[0.02, 0.02, 0.98, 0.95]
    )

    output_path = os.path.join(
        OUTPUT_DIR,
        filename,
    )

    fig.savefig(
        output_path,
        dpi=250,
        bbox_inches="tight",
        facecolor=BG,
    )

    plt.show()
    plt.close(fig)

    print(f"Saved: {output_path}")

# ============================================================
# 1. HIGHEST MEAN POPULARITY
# ============================================================

plot_triad_metric(
    data=triads_pdf,
    metric="mean_votes",
    title="Fragrance Accord Triads with the Highest Mean Popularity",
    xlabel="Mean Number of Votes per Fragrance",
    filename="accord-triads-highest-mean-popularity.png",
    bar_color=GOLD,
    value_format="number",
    ranking="highest",
)

# ============================================================
# 2. HIGHEST MEAN RATING
# ============================================================

plot_triad_metric(
    data=triads_pdf,
    metric="mean_rating",
    title="Fragrance Accord Triads with the Highest Mean Rating",
    xlabel="Mean Fragrance Rating",
    filename="accord-triads-highest-mean-rating.png",
    bar_color=ROSE,
    value_format="rating",
    ranking="highest",
)

# ============================================================
# 3. LOWEST MEAN POPULARITY
# ============================================================

plot_triad_metric(
    data=triads_pdf,
    metric="mean_votes",
    title="Fragrance Accord Triads with the Lowest Mean Popularity",
    xlabel="Mean Number of Votes per Fragrance",
    filename="accord-triads-lowest-mean-popularity.png",
    bar_color=BROWN,
    value_format="number",
    ranking="lowest",
)

# ============================================================
# 4. LOWEST MEAN RATING
# ============================================================

plot_triad_metric(
    data=triads_pdf,
    metric="mean_rating",
    title="Fragrance Accord Triads with the Lowest Mean Rating",
    xlabel="Mean Fragrance Rating",
    filename="accord-triads-lowest-mean-rating.png",
    bar_color="#80606b",
    value_format="rating",
    ranking="lowest",
)

print("\nAll four triad charts have been generated.")